# NB14 — Controlled Graph Refinement Follow-up

## Ρόλος του notebook

Το `NB14` αποτελεί ένα **controlled graph refinement follow-up** πάνω στα canonical graph artifacts και στα ήδη υλοποιημένα `NB12` και `NB13`.

Το notebook παραμένει αυστηρά:

- **forecasting-first**
- **benchmark-safe**
- **non-overclaiming**
- **local by default**

Δεν εισάγει νέο broad graph benchmark suite, δεν αλλάζει το upstream data contract και δεν επαναορίζει το canonical baseline benchmark.

## Κεντρικό refinement question

Το παρόν notebook εξετάζει **ένα μόνο κεντρικό graph refinement question**:

> είναι το μικρό πλεονέκτημα της `local_pruned_graph` οικογένειας αρκετά σταθερό όταν μεταβάλλεται ελεγχόμενα η ένταση του pruning, χωρίς να αλλάζει η model family και χωρίς να σπάει το `NB12` / `NB13` training and reporting contract;

## Αυστηρό comparison boundary

Οι συγκρίσεις του `NB14` περιορίζονται αυστηρά σε:

- το `NB12` reference
- το best configuration του `NB13`

Το `NB14` δεν χρησιμοποιείται για claims υπεροχής των graph models έναντι του συνολικού benchmark backbone και δεν επεκτείνεται σε sequence models, Mamba / Graph-Mamba, PHM implementation ή digital twin application work.

## Μεθοδολογικός κανόνας

Σε όλο το notebook διατηρείται ο ίδιος benchmark-safe κανόνας:

- **validation only for model selection**
- **test only for final reporting**

Στο τέλος του notebook θα εκτελεστεί και **final sanity test** ώστε να επιβεβαιωθεί ότι το refinement experiment παρέμεινε συμβατό με το established benchmark contract.

In [1]:
# NB14 | Environment bootstrap, path resolution και deterministic setup

from pathlib import Path
import json
import random
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


def find_repo_root(start_dir: Path) -> Path:
    """
    Εντοπίζει το repository root ψάχνοντας προς τα πάνω
    μέχρι να βρει το canonical src/config.py.
    """
    current = start_dir.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate

    raise FileNotFoundError(
        "Δεν βρέθηκε το repository root με βάση το canonical src/config.py."
    )


def seed_everything(seed: int) -> None:
    """
    Ρυθμίζει deterministic behavior όσο γίνεται για local reruns.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# Κρατάμε το notebook output καθαρό και ελεγχόμενο.
warnings.filterwarnings("ignore", category=FutureWarning)

# Εντοπισμός repository root ανεξάρτητα από το πού εκτελείται το notebook.
REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.config as project_config  # noqa: E402


# Χρησιμοποιούμε το canonical seed του project όπου είναι διαθέσιμο.
SEED = int(getattr(project_config, "RANDOM_SEED", 42))
seed_everything(SEED)

# Local-by-default runtime device.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Προαιρετική βασική μορφοποίηση για plots αργότερα.
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

print("NB14 environment bootstrap completed.")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"SEED: {SEED}")
print(f"DEVICE: {DEVICE}")

NB14 environment bootstrap completed.
REPO_ROOT: C:\Users\diony\Desktop\WindPower_DigitalTwin
SEED: 42
DEVICE: cpu


## Canonical input contract και read-only reference policy

Το `NB14` δεν ξεκινά από raw data, ούτε ξανακάνει feature engineering, temporal split ή graph packaging.  
Πατάει πάνω στο ήδη σταθεροποιημένο graph artifact chain του repository.

### Primary runtime inputs

Τα primary runtime inputs του notebook είναι τα canonical packaged graph artifacts του `NB11`, δηλαδή τα train / validation / test graph datasets και τα σχετικά manifests / summaries που περιγράφουν το packaging contract.

### Read-only reference artifacts

Για comparison και interpretation, το `NB14` χρησιμοποιεί μόνο ως **read-only references**:

- το `NB12` reference run
- το best configuration του `NB13`
- τα αντίστοιχα compact configs / summaries / metrics exports

Αυτά τα reference artifacts δεν τροποποιούνται και δεν αντιμετωπίζονται ως νέα training inputs.

### Benchmark boundary

Το canonical tabular benchmark artifact του repository παραμένει το:

`data/processed/baseline_metrics.csv`

Το `NB14` μπορεί να το διαβάζει μόνο ως benchmark context.  
Δεν το επαναορίζει, δεν το ενημερώνει και δεν το μετατρέπει σε graph benchmark table.

### Scope discipline

Στο παρόν notebook:

- δεν αλλάζει το upstream data contract
- δεν αλλάζει η baseline ladder
- δεν ανοίγει νέο model family
- δεν εισάγεται νέο broad benchmark suite
- δεν χρησιμοποιείται το test split για model selection

Άρα το `NB14` πρέπει να ερμηνεύεται ως **controlled refinement stage** πάνω στο υπάρχον `NB11 -> NB12 -> NB13` chain και όχι ως νέο αυτόνομο benchmarking branch.

In [2]:
# NB14 | Fail-fast existence checks για canonical upstream artifacts

from pathlib import Path


def assert_paths_exist(path_map: dict[str, Path], group_name: str) -> None:
    """
    Fail-fast έλεγχος ότι όλα τα required artifacts ενός group υπάρχουν.
    """
    missing = {name: path for name, path in path_map.items() if not path.exists()}

    if missing:
        missing_lines = "\n".join(
            f"- {name}: {path}" for name, path in missing.items()
        )
        raise FileNotFoundError(
            f"Λείπουν required artifacts από το group '{group_name}':\n{missing_lines}"
        )

    print(f"[OK] {group_name}: {len(path_map)} required artifacts found.")


PRIMARY_RUNTIME_PATHS = {
    "NB11_FEATURE_ROLE_MANIFEST": project_config.NB11_FEATURE_ROLE_MANIFEST,
    "NB11_NODE_FEATURE_MANIFEST": project_config.NB11_NODE_FEATURE_MANIFEST,
    "NB11_SPLIT_GRAPH_PACKAGING_SUMMARY": project_config.NB11_SPLIT_GRAPH_PACKAGING_SUMMARY,
    "NB11_PACKAGING_STATUS_MANIFEST": project_config.NB11_PACKAGING_STATUS_MANIFEST,
    "NB11_TRAIN_TIMESTAMP_COVERAGE": project_config.NB11_TRAIN_TIMESTAMP_COVERAGE,
    "NB11_VAL_TIMESTAMP_COVERAGE": project_config.NB11_VAL_TIMESTAMP_COVERAGE,
    "NB11_TEST_TIMESTAMP_COVERAGE": project_config.NB11_TEST_TIMESTAMP_COVERAGE,
    "NB11_TRAIN_GRAPH_DATASET": project_config.NB11_TRAIN_GRAPH_DATASET,
    "NB11_VAL_GRAPH_DATASET": project_config.NB11_VAL_GRAPH_DATASET,
    "NB11_TEST_GRAPH_DATASET": project_config.NB11_TEST_GRAPH_DATASET,
}

READ_ONLY_REFERENCE_PATHS = {
    "BASELINE_METRICS_PATH": project_config.BASELINE_METRICS_PATH,
    "NB12_RUN_CONFIG": project_config.NB12_RUN_CONFIG,
    "NB12_TEST_METRICS": project_config.NB12_TEST_METRICS,
    "NB12_TRAINING_HISTORY": project_config.NB12_TRAINING_HISTORY,
    "NB12_BENCHMARK_COMPARISON": project_config.NB12_BENCHMARK_COMPARISON,
    "NB13_RUN_CONFIG": project_config.NB13_RUN_CONFIG,
    "NB13_VALIDATION_SUMMARY": project_config.NB13_VALIDATION_SUMMARY,
    "NB13_TEST_METRICS": project_config.NB13_TEST_METRICS,
    "NB13_ABLATION_COMPARISON": project_config.NB13_ABLATION_COMPARISON,
}

assert_paths_exist(PRIMARY_RUNTIME_PATHS, "PRIMARY_RUNTIME_PATHS")
assert_paths_exist(READ_ONLY_REFERENCE_PATHS, "READ_ONLY_REFERENCE_PATHS")

print("\nPrimary runtime artifacts:")
for name, path in PRIMARY_RUNTIME_PATHS.items():
    print(f"- {name}: {path}")

print("\nRead-only reference artifacts:")
for name, path in READ_ONLY_REFERENCE_PATHS.items():
    print(f"- {name}: {path}")

[OK] PRIMARY_RUNTIME_PATHS: 10 required artifacts found.
[OK] READ_ONLY_REFERENCE_PATHS: 9 required artifacts found.

Primary runtime artifacts:
- NB11_FEATURE_ROLE_MANIFEST: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging\nb11_feature_role_manifest.csv
- NB11_NODE_FEATURE_MANIFEST: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging\nb11_node_feature_manifest.csv
- NB11_SPLIT_GRAPH_PACKAGING_SUMMARY: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging\nb11_split_graph_packaging_summary.csv
- NB11_PACKAGING_STATUS_MANIFEST: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging\nb11_packaging_status_manifest.csv
- NB11_TRAIN_TIMESTAMP_COVERAGE: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging\nb11_train_timestamp_coverage.csv
- NB11_VAL_TIMESTAMP_COVERAGE: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging\nb11_val_timestamp_coverage.csv
-

## Frozen contract από `NB12` και `NB13`

Το `NB14` δεν ξεκινά νέο graph modeling branch.  
Λειτουργεί ως **controlled refinement follow-up** πάνω στο ήδη υπάρχον contract των `NB12` και `NB13`.

### Τι παραμένει frozen

Στο `NB14` παραμένουν σταθερά τα εξής:

- η ίδια general model family
- η ίδια benchmark-safe λογική train / validation / test
- το canonical packaged input contract του `NB11`
- η ίδια βασική training philosophy του `NB12`
- το validation-only selection rule
- το test-only final reporting rule
- το strict comparison boundary μόνο με `NB12` reference και `NB13` best run

### Τι δεν αλλάζουμε

Το notebook δεν αλλάζει:

- upstream preprocessing
- temporal split boundaries
- target definition
- canonical baseline benchmark artifact
- broader graph design space
- model family
- research framing του repository

### Τι επιτρέπεται να αλλάξει

Το μόνο επιτρεπτό change space του `NB14` είναι ένα **στενά ορισμένο refinement dimension** μέσα στο already-supported post-`NB13` graph space.

Στόχος δεν είναι να εισαχθεί νέα αρχιτεκτονική, αλλά να ελεγχθεί αν ένα μικρό topology-related refinement παραμένει συνεπές όταν όλα τα υπόλοιπα κρίσιμα στοιχεία μένουν frozen.

### Ερμηνευτικό όριο

Άρα το `NB14` πρέπει να διαβαστεί ως:

- controlled refinement stage
- post-`NB13` follow-up
- cautious graph evidence extension

και όχι ως:

- νέο broad graph benchmark
- graph superiority validation stage
- sequence-model notebook
- PHM implementation notebook

In [3]:
# NB14 | Load compact manifests, summaries και read-only reference configs

def load_json_file(path: Path) -> dict:
    """
    Φορτώνει JSON artifact με explicit UTF-8 handling.
    """
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_csv_file(path: Path) -> pd.DataFrame:
    """
    Φορτώνει CSV artifact και αποτυγχάνει νωρίς αν το αρχείο είναι κενό.
    """
    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(f"Το CSV artifact είναι κενό: {path}")

    return df


# ------------------------------------------------------------------
# NB11 compact manifests / summaries
# ------------------------------------------------------------------
nb11_feature_role_manifest_df = load_csv_file(project_config.NB11_FEATURE_ROLE_MANIFEST)
nb11_node_feature_manifest_df = load_csv_file(project_config.NB11_NODE_FEATURE_MANIFEST)
nb11_split_graph_packaging_summary_df = load_csv_file(project_config.NB11_SPLIT_GRAPH_PACKAGING_SUMMARY)
nb11_packaging_status_manifest_df = load_csv_file(project_config.NB11_PACKAGING_STATUS_MANIFEST)

nb11_train_timestamp_coverage_df = load_csv_file(project_config.NB11_TRAIN_TIMESTAMP_COVERAGE)
nb11_val_timestamp_coverage_df = load_csv_file(project_config.NB11_VAL_TIMESTAMP_COVERAGE)
nb11_test_timestamp_coverage_df = load_csv_file(project_config.NB11_TEST_TIMESTAMP_COVERAGE)

# ------------------------------------------------------------------
# Canonical benchmark context (read-only)
# ------------------------------------------------------------------
baseline_metrics_df = load_csv_file(project_config.BASELINE_METRICS_PATH)

# ------------------------------------------------------------------
# NB12 reference artifacts (read-only)
# ------------------------------------------------------------------
nb12_run_config = load_json_file(project_config.NB12_RUN_CONFIG)
nb12_test_metrics_df = load_csv_file(project_config.NB12_TEST_METRICS)
nb12_training_history_df = load_csv_file(project_config.NB12_TRAINING_HISTORY)
nb12_benchmark_comparison_df = load_csv_file(project_config.NB12_BENCHMARK_COMPARISON)

# ------------------------------------------------------------------
# NB13 reference artifacts (read-only)
# ------------------------------------------------------------------
nb13_run_config = load_json_file(project_config.NB13_RUN_CONFIG)
nb13_validation_summary_df = load_csv_file(project_config.NB13_VALIDATION_SUMMARY)
nb13_test_metrics_df = load_csv_file(project_config.NB13_TEST_METRICS)
nb13_ablation_comparison_df = load_csv_file(project_config.NB13_ABLATION_COMPARISON)

# ------------------------------------------------------------------
# Compact audit printout
# ------------------------------------------------------------------
loaded_tables = {
    "nb11_feature_role_manifest_df": nb11_feature_role_manifest_df,
    "nb11_node_feature_manifest_df": nb11_node_feature_manifest_df,
    "nb11_split_graph_packaging_summary_df": nb11_split_graph_packaging_summary_df,
    "nb11_packaging_status_manifest_df": nb11_packaging_status_manifest_df,
    "nb11_train_timestamp_coverage_df": nb11_train_timestamp_coverage_df,
    "nb11_val_timestamp_coverage_df": nb11_val_timestamp_coverage_df,
    "nb11_test_timestamp_coverage_df": nb11_test_timestamp_coverage_df,
    "baseline_metrics_df": baseline_metrics_df,
    "nb12_test_metrics_df": nb12_test_metrics_df,
    "nb12_training_history_df": nb12_training_history_df,
    "nb12_benchmark_comparison_df": nb12_benchmark_comparison_df,
    "nb13_validation_summary_df": nb13_validation_summary_df,
    "nb13_test_metrics_df": nb13_test_metrics_df,
    "nb13_ablation_comparison_df": nb13_ablation_comparison_df,
}

print("Loaded tabular artifacts:")
for name, df in loaded_tables.items():
    print(f"- {name}: shape={df.shape}")

print("\nLoaded JSON artifacts:")
print(f"- nb12_run_config keys: {sorted(nb12_run_config.keys())}")
print(f"- nb13_run_config keys: {sorted(nb13_run_config.keys())}")

print("\nCompact reference snapshot:")
print(f"- NB12 seed: {nb12_run_config.get('seed', 'N/A')}")
print(f"- NB12 device: {nb12_run_config.get('device', 'N/A')}")
print(f"- NB13 notebook role: {nb13_run_config.get('role', 'N/A')}")
print(f"- NB13 selection rule: {nb13_run_config.get('selection_rule', 'N/A')}")
print(f"- NB13 reporting rule: {nb13_run_config.get('reporting_rule', 'N/A')}")

Loaded tabular artifacts:
- nb11_feature_role_manifest_df: shape=(48, 6)
- nb11_node_feature_manifest_df: shape=(48, 5)
- nb11_split_graph_packaging_summary_df: shape=(3, 18)
- nb11_packaging_status_manifest_df: shape=(10, 3)
- nb11_train_timestamp_coverage_df: shape=(7819, 4)
- nb11_val_timestamp_coverage_df: shape=(720, 4)
- nb11_test_timestamp_coverage_df: shape=(4295, 4)
- baseline_metrics_df: shape=(5, 4)
- nb12_test_metrics_df: shape=(1, 5)
- nb12_training_history_df: shape=(11, 8)
- nb12_benchmark_comparison_df: shape=(6, 4)
- nb13_validation_summary_df: shape=(6, 13)
- nb13_test_metrics_df: shape=(6, 11)
- nb13_ablation_comparison_df: shape=(6, 21)

Loaded JSON artifacts:
- nb12_run_config keys: ['best_epoch', 'best_val_loss', 'device', 'input_feature_dim', 'n_nodes', 'n_test_snapshots', 'n_train_snapshots', 'n_val_snapshots', 'seed', 'training_config']
- nb13_run_config keys: ['ablation_dimensions', 'benchmark_reference_mode', 'frozen_dimensions', 'notebook', 'notes', 'planned

## Controlled refinement plan

Με βάση το `NB13`, το `NB14` δεν ανοίγει νέο topology search space.  
Ακολουθεί ένα **στενά περιορισμένο refinement plan**.

### Refinement family

Το refinement του `NB14` περιορίζεται μόνο στην οικογένεια:

- `local_pruned_graph`

Η επιλογή αυτή γίνεται επειδή στο `NB13` η συγκεκριμένη οικογένεια έδωσε την καλύτερη επίδοση μέσα στο controlled ablation space, αλλά με μικρό observed gain που απαιτεί cautious follow-up και όχι strong superiority claim.

### Μοναδική refinement διάσταση

Το `NB14` θα ελέγξει μόνο μία κεντρική διάσταση:

- **pruning strength**, δηλαδή πόσο επιθετικά ή ήπια διατηρούνται οι ακμές μέσα στο local-pruned topology

Η διερεύνηση αυτή θα γίνει με **predeclared candidate settings** και όχι με ανοιχτό-ended tuning.

### Τι δεν θα εξεταστεί εδώ

Στο `NB14` δεν θα εξεταστούν:

- νέα graph family
- νέα GNN architecture
- νέα message-passing philosophy
- νέο temporal modeling branch
- sequence models ή Mamba-family designs
- broader graph redesign

### Ερμηνεία του refinement

Το ζητούμενο δεν είναι να αποδειχθεί ότι ένα graph setup είναι συνολικά superior.  
Το ζητούμενο είναι να ελεγχθεί αν ένα μικρό και benchmark-safe refinement γύρω από το καλύτερο `NB13` family παραμένει μεθοδολογικά καθαρό και εμπειρικά συνεπές.

In [5]:
# NB14 | Corrected predeclared controlled refinement registry
# Χρησιμοποιούμε:
# - NB13_EXPERIMENT_REGISTRY για topology metadata
# - NB13_VALIDATION_SUMMARY για validation-side ranking

def require_columns(df: pd.DataFrame, required_columns: list[str], df_name: str) -> None:
    """
    Fail-fast έλεγχος ότι ένα DataFrame περιέχει τις απαιτούμενες στήλες.
    """
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise KeyError(
            f"Το DataFrame '{df_name}' δεν περιέχει τις required στήλες: {missing}"
        )


# ------------------------------------------------------------------
# Load NB13 experiment registry (topology metadata source)
# ------------------------------------------------------------------
if not project_config.NB13_EXPERIMENT_REGISTRY.exists():
    raise FileNotFoundError(
        "Δεν βρέθηκε το NB13_EXPERIMENT_REGISTRY. "
        "Το NB14 χρειάζεται αυτό το artifact για να ανακτήσει "
        "edge_policy και edge_keep_quantile χωρίς schema assumptions."
    )

nb13_experiment_registry_df = load_csv_file(project_config.NB13_EXPERIMENT_REGISTRY)

require_columns(
    nb13_experiment_registry_df,
    [
        "experiment_id",
        "topology_variant",
        "edge_policy",
        "edge_keep_quantile",
        "message_passing_layers",
    ],
    "nb13_experiment_registry_df",
)

require_columns(
    nb13_validation_summary_df,
    [
        "experiment_id",
        "topology_variant",
        "message_passing_layers",
        "best_val_loss",
    ],
    "nb13_validation_summary_df",
)

# ------------------------------------------------------------------
# Merge registry metadata with validation ranking information
# ------------------------------------------------------------------
nb13_registry_enriched_df = nb13_experiment_registry_df.merge(
    nb13_validation_summary_df[
        [
            "experiment_id",
            "topology_variant",
            "message_passing_layers",
            "best_val_loss",
        ]
    ],
    on=["experiment_id", "topology_variant", "message_passing_layers"],
    how="left",
    validate="one_to_one",
)

if nb13_registry_enriched_df["best_val_loss"].isna().any():
    missing_best_val = nb13_registry_enriched_df.loc[
        nb13_registry_enriched_df["best_val_loss"].isna(),
        "experiment_id",
    ].tolist()
    raise ValueError(
        "Κάποια NB13 registry rows δεν μπόρεσαν να αντιστοιχιστούν "
        f"με validation losses: {missing_best_val}"
    )

# Κρατάμε μόνο την local-pruned family, γιατί αυτή είναι το refinement target του NB14.
nb13_local_pruned_df = (
    nb13_registry_enriched_df.loc[
        nb13_registry_enriched_df["topology_variant"] == "local_pruned_graph"
    ]
    .copy()
)

if nb13_local_pruned_df.empty:
    raise ValueError(
        "Δεν βρέθηκαν local_pruned_graph runs στο merged NB13 registry/validation space."
    )

# Deterministic validation-side reference selection.
nb13_local_pruned_df = nb13_local_pruned_df.sort_values(
    by=["best_val_loss", "edge_keep_quantile", "experiment_id"],
    ascending=[True, True, True],
    kind="mergesort",
).reset_index(drop=True)

nb13_local_pruned_reference_row = nb13_local_pruned_df.iloc[0].copy()

NB13_REFERENCE_EXPERIMENT_ID = str(nb13_local_pruned_reference_row["experiment_id"])
NB13_REFERENCE_EDGE_POLICY = str(nb13_local_pruned_reference_row["edge_policy"])
NB13_REFERENCE_MESSAGE_PASSING_LAYERS = int(
    nb13_local_pruned_reference_row["message_passing_layers"]
)
NB13_REFERENCE_EDGE_KEEP_QUANTILE = round(
    float(nb13_local_pruned_reference_row["edge_keep_quantile"]),
    2,
)
NB13_REFERENCE_BEST_VAL_LOSS = float(nb13_local_pruned_reference_row["best_val_loss"])

# Controlled neighborhood γύρω από το validation-side local-pruned reference.
quantile_offsets = (-0.10, -0.05, 0.00, 0.05, 0.10)

candidate_quantiles = sorted(
    {
        round(
            min(0.98, max(0.50, NB13_REFERENCE_EDGE_KEEP_QUANTILE + offset)),
            2,
        )
        for offset in quantile_offsets
    }
)

if len(candidate_quantiles) < 3:
    raise ValueError(
        "Το candidate quantile set είναι υπερβολικά μικρό μετά το clipping. "
        "Χρειάζονται τουλάχιστον 3 distinct candidates."
    )

NB14_EXPERIMENT_REGISTRY = pd.DataFrame(
    [
        {
            "experiment_id": f"NB14_E{i:02d}",
            "topology_variant": "local_pruned_graph",
            "edge_policy": NB13_REFERENCE_EDGE_POLICY,
            "edge_keep_quantile": quantile,
            "message_passing_layers": NB13_REFERENCE_MESSAGE_PASSING_LAYERS,
            "registry_source": "nb13_local_pruned_validation_ranked_reference",
            "reference_nb13_experiment_id": NB13_REFERENCE_EXPERIMENT_ID,
            "is_nb13_reference_quantile": quantile == NB13_REFERENCE_EDGE_KEEP_QUANTILE,
        }
        for i, quantile in enumerate(candidate_quantiles, start=1)
    ]
)

print("NB13 local-pruned validation reference:")
print(f"- experiment_id: {NB13_REFERENCE_EXPERIMENT_ID}")
print(f"- edge_policy: {NB13_REFERENCE_EDGE_POLICY}")
print(f"- message_passing_layers: {NB13_REFERENCE_MESSAGE_PASSING_LAYERS}")
print(f"- edge_keep_quantile: {NB13_REFERENCE_EDGE_KEEP_QUANTILE:.2f}")
print(f"- best_val_loss: {NB13_REFERENCE_BEST_VAL_LOSS:.6f}")

print("\nNB14 predeclared candidate quantiles:")
print(candidate_quantiles)

print("\nNB14 experiment registry:")
NB14_EXPERIMENT_REGISTRY

NB13 local-pruned validation reference:
- experiment_id: NB13_E06
- edge_policy: keep_shortest_edges_only
- message_passing_layers: 2
- edge_keep_quantile: 0.50
- best_val_loss: 0.060495

NB14 predeclared candidate quantiles:
[0.5, 0.55, 0.6]

NB14 experiment registry:


,experiment_id,topology_variant,edge_policy,edge_keep_quantile,message_passing_layers,registry_source,reference_nb13_experiment_id,is_nb13_reference_quantile
0,NB14_E01,local_pruned_graph,keep_shortest_edges_only,0.50,2,nb13_local_pruned_validation_ranked_reference,NB13_E06,True
1,NB14_E02,local_pruned_graph,keep_shortest_edges_only,0.55,2,nb13_local_pruned_validation_ranked_reference,NB13_E06,False
2,NB14_E03,local_pruned_graph,keep_shortest_edges_only,0.60,2,nb13_local_pruned_validation_ranked_reference,NB13_E06,False


## Deterministic pruning policy και graph-construction boundary

Το `NB14` δεν επανασχεδιάζει το συνολικό spatial graph του repository.  
Δεν επιστρέφει στο `NB04`, δεν ξανακάνει graph-readiness verification τύπου `NB10` και δεν ξανακάνει packaging τύπου `NB11`.

### Τι κάνει εδώ το notebook

Το notebook εφαρμόζει μόνο ένα **deterministic local pruning refinement** πάνω σε ήδη συμβατό graph space, με anchor το best local-pruned validation reference του `NB13`.

Η λογική είναι η εξής:

- κρατάμε σταθερή την `local_pruned_graph` family
- κρατάμε σταθερό το `edge_policy`
- κρατάμε σταθερό το message-passing depth
- μεταβάλλουμε μόνο το `edge_keep_quantile`

### Γιατί αυτό είναι μεθοδολογικά σωστό

Αυτό το design επιτρέπει να εξεταστεί η επίδραση του pruning strength χωρίς να συγχέεται το αποτέλεσμα με:

- νέο graph construction rule
- νέο topology family
- αλλαγή στο upstream spatial contract
- αλλαγή στο packaged dataset contract

Άρα η μεταβολή που μετράμε παραμένει όσο γίνεται πιο καθαρή και ερμηνεύσιμη.

### Τι δεν επιτρέπεται εδώ

Στο `NB14` δεν επιτρέπεται:

- ad hoc manual edge editing
- introduction νέων node relations
- αλλαγή στο node ordering
- αλλαγή στο feature space
- αλλαγή στο snapshot construction logic
- retrospective χρήση test information για να οριστεί το refined topology

Η refined topology πρέπει να προκύπτει μόνο από **predeclared rules** και να διατηρεί benchmark-safe separation ανάμεσα σε selection και final reporting.

In [6]:
# NB14 | Build deterministic refined edge bundles from canonical graph artifacts

def load_numpy_array(path: Path, expected_ndim: int | None = None) -> np.ndarray:
    """
    Φορτώνει .npy artifact με προαιρετικό έλεγχο dimensionality.
    """
    if not path.exists():
        raise FileNotFoundError(f"Δεν βρέθηκε το numpy artifact: {path}")

    arr = np.load(path)

    if expected_ndim is not None and arr.ndim != expected_ndim:
        raise ValueError(
            f"Το artifact {path.name} έχει ndim={arr.ndim}, "
            f"ενώ αναμενόταν ndim={expected_ndim}."
        )

    return arr


def normalize_edge_index_array(edge_index_arr: np.ndarray) -> np.ndarray:
    """
    Κανονικοποιεί το edge_index σε shape [2, E].
    Επιτρέπεται input shape είτε [2, E] είτε [E, 2].
    """
    if edge_index_arr.ndim != 2:
        raise ValueError(
            f"Το edge_index πρέπει να είναι 2D array, όχι ndim={edge_index_arr.ndim}."
        )

    if edge_index_arr.shape[0] == 2:
        normalized = edge_index_arr
    elif edge_index_arr.shape[1] == 2:
        normalized = edge_index_arr.T
    else:
        raise ValueError(
            "Το edge_index δεν έχει συμβατό shape. "
            f"Βρέθηκε shape={edge_index_arr.shape}, αναμενόταν [2, E] ή [E, 2]."
        )

    return normalized.astype(np.int64, copy=False)


def validate_distance_matrix(distance_matrix: np.ndarray) -> None:
    """
    Fail-fast checks για canonical distance matrix.
    """
    if distance_matrix.ndim != 2:
        raise ValueError("Η distance matrix πρέπει να είναι 2D.")

    if distance_matrix.shape[0] != distance_matrix.shape[1]:
        raise ValueError(
            "Η distance matrix πρέπει να είναι square. "
            f"Βρέθηκε shape={distance_matrix.shape}."
        )

    if not np.allclose(distance_matrix, distance_matrix.T, atol=1e-10):
        raise ValueError("Η distance matrix δεν είναι συμμετρική.")

    if not np.allclose(np.diag(distance_matrix), 0.0, atol=1e-10):
        raise ValueError("Η distance matrix δεν έχει zero diagonal.")


def build_undirected_edge_table(
    edge_index_arr: np.ndarray,
    distance_matrix: np.ndarray,
) -> pd.DataFrame:
    """
    Μετατρέπει directed edge_index σε unique undirected edge table
    και συνδέει κάθε ακμή με τη γεωγραφική της απόσταση.
    """
    src = edge_index_arr[0]
    dst = edge_index_arr[1]

    if len(src) != len(dst):
        raise ValueError("Το edge_index έχει ασύμβατο src/dst length.")

    directed_df = pd.DataFrame({"src": src, "dst": dst})

    if (directed_df["src"] == directed_df["dst"]).any():
        raise ValueError("Το canonical edge_index περιέχει self-loops, κάτι μη αναμενόμενο.")

    directed_df["u"] = directed_df[["src", "dst"]].min(axis=1)
    directed_df["v"] = directed_df[["src", "dst"]].max(axis=1)

    undirected_df = (
        directed_df[["u", "v"]]
        .drop_duplicates()
        .sort_values(["u", "v"], kind="mergesort")
        .reset_index(drop=True)
    )

    undirected_df["distance_km"] = [
        float(distance_matrix[u, v]) for u, v in undirected_df[["u", "v"]].itertuples(index=False)
    ]

    if (undirected_df["distance_km"] < 0).any():
        raise ValueError("Βρέθηκαν αρνητικές αποστάσεις στο undirected edge table.")

    if undirected_df["distance_km"].isna().any():
        raise ValueError("Βρέθηκαν NaN distances στο undirected edge table.")

    return undirected_df


def make_directed_edge_index_from_undirected(undirected_edges_df: pd.DataFrame) -> np.ndarray:
    """
    Παράγει directed edge_index [2, E] από unique undirected edges.
    Για κάθε (u, v) κρατάμε και τις δύο κατευθύνσεις.
    """
    forward_edges = undirected_edges_df[["u", "v"]].to_numpy(dtype=np.int64)
    reverse_edges = undirected_edges_df[["v", "u"]].to_numpy(dtype=np.int64)

    directed_edges = np.vstack([forward_edges, reverse_edges])
    directed_edges = directed_edges[np.lexsort((directed_edges[:, 1], directed_edges[:, 0]))]

    return directed_edges.T


def build_pruned_edge_bundle(
    undirected_edge_table: pd.DataFrame,
    edge_keep_quantile: float,
    edge_policy: str,
) -> dict:
    """
    Χτίζει deterministic pruned topology bundle.
    Προς το παρόν το NB14 υποστηρίζει μόνο keep_shortest_edges_only.
    """
    if edge_policy != "keep_shortest_edges_only":
        raise NotImplementedError(
            f"Μη υποστηριζόμενο edge_policy για το NB14: {edge_policy}"
        )

    if not (0.0 < edge_keep_quantile <= 1.0):
        raise ValueError(
            f"Το edge_keep_quantile πρέπει να είναι στο (0, 1], όχι {edge_keep_quantile}."
        )

    n_total_undirected_edges = len(undirected_edge_table)
    n_keep_undirected_edges = max(1, int(np.ceil(edge_keep_quantile * n_total_undirected_edges)))

    kept_undirected_df = (
        undirected_edge_table
        .sort_values(["distance_km", "u", "v"], ascending=[True, True, True], kind="mergesort")
        .head(n_keep_undirected_edges)
        .reset_index(drop=True)
    )

    pruned_edge_index = make_directed_edge_index_from_undirected(kept_undirected_df)

    return {
        "edge_keep_quantile": round(float(edge_keep_quantile), 2),
        "edge_policy": edge_policy,
        "n_total_undirected_edges": int(n_total_undirected_edges),
        "n_kept_undirected_edges": int(len(kept_undirected_df)),
        "n_kept_directed_edges": int(pruned_edge_index.shape[1]),
        "max_kept_distance_km": float(kept_undirected_df["distance_km"].max()),
        "mean_kept_distance_km": float(kept_undirected_df["distance_km"].mean()),
        "kept_undirected_edges_df": kept_undirected_df,
        "pruned_edge_index": pruned_edge_index,
    }


# ------------------------------------------------------------------
# Load canonical graph backbone
# ------------------------------------------------------------------
canonical_edge_index_arr = normalize_edge_index_array(
    load_numpy_array(project_config.GRAPH_EDGE_INDEX_PATH, expected_ndim=2)
)
canonical_distance_matrix = load_numpy_array(
    project_config.GRAPH_DISTANCE_MATRIX_PATH,
    expected_ndim=2,
)

validate_distance_matrix(canonical_distance_matrix)

n_nodes_from_distance_matrix = canonical_distance_matrix.shape[0]

if canonical_edge_index_arr.min() < 0:
    raise ValueError("Το canonical edge_index περιέχει αρνητικά node indices.")

if canonical_edge_index_arr.max() >= n_nodes_from_distance_matrix:
    raise ValueError(
        "Το canonical edge_index περιέχει node index εκτός distance-matrix range."
    )

if n_nodes_from_distance_matrix != int(nb12_run_config["n_nodes"]):
    raise ValueError(
        "Ασυμφωνία ανάμεσα σε canonical distance-matrix node count "
        f"({n_nodes_from_distance_matrix}) και NB12 n_nodes ({nb12_run_config['n_nodes']})."
    )

canonical_undirected_edge_table_df = build_undirected_edge_table(
    canonical_edge_index_arr,
    canonical_distance_matrix,
)

# ------------------------------------------------------------------
# Build refined edge bundles for every NB14 candidate
# ------------------------------------------------------------------
NB14_EDGE_BUNDLES = {}

for row in NB14_EXPERIMENT_REGISTRY.itertuples(index=False):
    bundle = build_pruned_edge_bundle(
        undirected_edge_table=canonical_undirected_edge_table_df,
        edge_keep_quantile=float(row.edge_keep_quantile),
        edge_policy=str(row.edge_policy),
    )

    NB14_EDGE_BUNDLES[row.experiment_id] = bundle

# ------------------------------------------------------------------
# Compact summary table
# ------------------------------------------------------------------
nb14_edge_bundle_summary_df = pd.DataFrame(
    [
        {
            "experiment_id": experiment_id,
            "edge_policy": bundle["edge_policy"],
            "edge_keep_quantile": bundle["edge_keep_quantile"],
            "n_total_undirected_edges": bundle["n_total_undirected_edges"],
            "n_kept_undirected_edges": bundle["n_kept_undirected_edges"],
            "n_kept_directed_edges": bundle["n_kept_directed_edges"],
            "max_kept_distance_km": bundle["max_kept_distance_km"],
            "mean_kept_distance_km": bundle["mean_kept_distance_km"],
        }
        for experiment_id, bundle in NB14_EDGE_BUNDLES.items()
    ]
).sort_values(
    by=["edge_keep_quantile", "experiment_id"],
    ascending=[True, True],
    kind="mergesort",
).reset_index(drop=True)

print("Canonical graph backbone summary:")
print(f"- n_nodes: {n_nodes_from_distance_matrix}")
print(f"- canonical directed edges: {canonical_edge_index_arr.shape[1]}")
print(f"- canonical undirected edges: {len(canonical_undirected_edge_table_df)}")

print("\nNB14 refined edge bundle summary:")
nb14_edge_bundle_summary_df

Canonical graph backbone summary:
- n_nodes: 256
- canonical directed edges: 1068
- canonical undirected edges: 534

NB14 refined edge bundle summary:


,experiment_id,edge_policy,edge_keep_quantile,n_total_undirected_edges,n_kept_undirected_edges,n_kept_directed_edges,max_kept_distance_km,mean_kept_distance_km
0,NB14_E01,keep_shortest_edges_only,0.50,534,267,534,38.750343,27.850407
1,NB14_E02,keep_shortest_edges_only,0.55,534,294,588,39.649263,28.887530
2,NB14_E03,keep_shortest_edges_only,0.60,534,321,642,41.698155,29.874086


## Runtime dataset / DataLoader policy

Στο `NB14` τα packaged graph datasets του `NB11` παραμένουν η **canonical runtime βάση**.

### Τι παραμένει σταθερό σε επίπεδο dataset

Παραμένουν σταθερά:

- το train / validation / test split contract
- το snapshot structure
- το node ordering
- το feature space
- το target space
- τα masks και η observed-data λογική
- το general batching / loading pattern

### Πού εφαρμόζεται το refinement

Το refinement του `NB14` εφαρμόζεται μόνο στο **topology layer**.

Πρακτικά αυτό σημαίνει ότι:

- χρησιμοποιούμε τα ήδη packaged graph snapshots ως βάση
- δεν ξανακάνουμε packaging
- δεν ξαναχτίζουμε feature tensors
- δεν αλλάζουμε labels ή targets
- αντικαθιστούμε μόνο το runtime `edge_index` με το predeclared refined topology του κάθε candidate

### Γιατί αυτό είναι σημαντικό

Με αυτόν τον τρόπο, η σύγκριση μεταξύ των `NB14` candidates παραμένει όσο γίνεται πιο καθαρή:

- ίδιο dataset contract
- ίδιο feature / target content
- ίδιο split logic
- ίδια model family
- διαφορετικό μόνο το refined connectivity pattern

### Benchmark-safe συνέπεια

Άρα, αν προκύψει διαφορά στην validation ή στην τελική test επίδοση, αυτή ερμηνεύεται ως αποτέλεσμα ενός **controlled topology refinement** και όχι ως αποτέλεσμα νέου preprocessing, νέου packaging ή αλλαγής στο data interface.

In [8]:
# NB14 | Inspect actual structure of packaged NB11 .pt artifacts
# Στόχος: να δούμε το πραγματικό schema πριν χτίσουμε runtime loaders.

def inspect_object(name: str, obj, max_dict_keys: int = 20, max_list_items: int = 3) -> None:
    print(f"\n{name}")
    print("-" * len(name))
    print(f"type: {type(obj)}")

    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"n_keys: {len(keys)}")
        print(f"keys: {keys[:max_dict_keys]}")

        for key in keys[:max_dict_keys]:
            value = obj[key]
            print(f"  - key='{key}': type={type(value)}", end="")
            if hasattr(value, "shape"):
                print(f", shape={tuple(value.shape)}")
            elif isinstance(value, (list, tuple)):
                print(f", len={len(value)}")
            elif isinstance(value, dict):
                print(f", n_keys={len(value)}")
            else:
                print()

    elif isinstance(obj, (list, tuple)):
        print(f"len: {len(obj)}")
        for i, value in enumerate(obj[:max_list_items]):
            print(f"  - item[{i}]: type={type(value)}", end="")
            if hasattr(value, "shape"):
                print(f", shape={tuple(value.shape)}")
            elif hasattr(value, "keys"):
                try:
                    keys_attr = value.keys
                    keys = keys_attr() if callable(keys_attr) else keys_attr
                    print(f", keys={list(keys)}")
                except Exception:
                    print()
            else:
                print()

    else:
        if hasattr(obj, "shape"):
            print(f"shape: {tuple(obj.shape)}")
        elif hasattr(obj, "keys"):
            try:
                keys_attr = obj.keys
                keys = keys_attr() if callable(keys_attr) else keys_attr
                print(f"keys: {list(keys)}")
            except Exception:
                pass


train_payload = torch.load(project_config.NB11_TRAIN_GRAPH_DATASET, map_location="cpu", weights_only=False)
val_payload = torch.load(project_config.NB11_VAL_GRAPH_DATASET, map_location="cpu", weights_only=False)
test_payload = torch.load(project_config.NB11_TEST_GRAPH_DATASET, map_location="cpu", weights_only=False)

inspect_object("train_payload", train_payload)
inspect_object("val_payload", val_payload)
inspect_object("test_payload", test_payload)

# Αν το payload είναι dict, κάνουμε και 1 επίπεδο deeper inspection.
for payload_name, payload in [
    ("train_payload", train_payload),
    ("val_payload", val_payload),
    ("test_payload", test_payload),
]:
    if isinstance(payload, dict):
        for key, value in list(payload.items())[:10]:
            inspect_object(f"{payload_name}['{key}']", value, max_dict_keys=15, max_list_items=2)


train_payload
-------------
type: <class 'dict'>
n_keys: 22
keys: ['split_name', 'node_ids', 'timestamps', 'edge_index', 'edge_attr_km', 'static_x', 'dynamic_x', 'target_y', 'baseline_reference', 'observed_mask', 'static_feature_names', 'dynamic_feature_names', 'target_name', 'baseline_name', 'n_nodes', 'n_timestamps', 'n_static_features', 'n_dynamic_features', 'partial_timestamp_count', 'full_coverage_timestamp_count']
  - key='split_name': type=<class 'str'>
  - key='node_ids': type=<class 'list'>, len=256
  - key='timestamps': type=<class 'list'>, len=7819
  - key='edge_index': type=<class 'torch.Tensor'>, shape=(2, 1068)
  - key='edge_attr_km': type=<class 'torch.Tensor'>, shape=(1068, 1)
  - key='static_x': type=<class 'torch.Tensor'>, shape=(256, 5)
  - key='dynamic_x': type=<class 'torch.Tensor'>, shape=(7819, 256, 36)
  - key='target_y': type=<class 'torch.Tensor'>, shape=(7819, 256)
  - key='baseline_reference': type=<class 'torch.Tensor'>, shape=(7819, 256)
  - key='observed

In [10]:
# NB14 | Corrected runtime materialization from packaged NB11 dict payloads
# Χρησιμοποιούμε το actual packaged payload ως authority για το runtime input dim.

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


def torch_load_packaged_payload(path: Path, split_name: str) -> dict:
    """
    Φορτώνει packaged NB11 payload από .pt artifact.
    Αναμένουμε dict-based portable payload.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Δεν βρέθηκε το packaged payload για το split '{split_name}': {path}"
        )

    payload = torch.load(path, map_location="cpu", weights_only=False)

    if not isinstance(payload, dict):
        raise TypeError(
            f"Το packaged payload του split '{split_name}' πρέπει να είναι dict. "
            f"Βρέθηκε type={type(payload)}."
        )

    return payload


def require_payload_keys(payload: dict, required_keys: list[str], split_name: str) -> None:
    missing = [key for key in required_keys if key not in payload]
    if missing:
        raise KeyError(
            f"Το packaged payload του split '{split_name}' δεν περιέχει required keys: {missing}"
        )


def validate_packaged_payload(payload: dict, split_name: str) -> None:
    """
    Fail-fast validation του actual NB11 packaged schema.
    Εδώ εμπιστευόμαστε το payload schema ως runtime authority.
    """
    required_keys = [
        "split_name",
        "node_ids",
        "timestamps",
        "edge_index",
        "static_x",
        "dynamic_x",
        "target_y",
        "baseline_reference",
        "observed_mask",
        "n_nodes",
        "n_timestamps",
        "n_static_features",
        "n_dynamic_features",
    ]
    require_payload_keys(payload, required_keys, split_name)

    n_nodes = int(payload["n_nodes"])
    n_timestamps = int(payload["n_timestamps"])
    n_static_features = int(payload["n_static_features"])
    n_dynamic_features = int(payload["n_dynamic_features"])

    edge_index = payload["edge_index"]
    static_x = payload["static_x"]
    dynamic_x = payload["dynamic_x"]
    target_y = payload["target_y"]
    baseline_reference = payload["baseline_reference"]
    observed_mask = payload["observed_mask"]

    if not isinstance(edge_index, torch.Tensor) or edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError(
            f"Το edge_index του split '{split_name}' πρέπει να έχει shape [2, E]. "
            f"Βρέθηκε: {type(edge_index)} με shape={tuple(edge_index.shape) if hasattr(edge_index, 'shape') else None}"
        )

    if not isinstance(static_x, torch.Tensor) or tuple(static_x.shape) != (n_nodes, n_static_features):
        raise ValueError(
            f"Το static_x του split '{split_name}' δεν έχει αναμενόμενο shape "
            f"({n_nodes}, {n_static_features}). Βρέθηκε: {tuple(static_x.shape)}"
        )

    if not isinstance(dynamic_x, torch.Tensor) or tuple(dynamic_x.shape) != (n_timestamps, n_nodes, n_dynamic_features):
        raise ValueError(
            f"Το dynamic_x του split '{split_name}' δεν έχει αναμενόμενο shape "
            f"({n_timestamps}, {n_nodes}, {n_dynamic_features}). Βρέθηκε: {tuple(dynamic_x.shape)}"
        )

    if not isinstance(target_y, torch.Tensor) or tuple(target_y.shape) != (n_timestamps, n_nodes):
        raise ValueError(
            f"Το target_y του split '{split_name}' δεν έχει αναμενόμενο shape "
            f"({n_timestamps}, {n_nodes}). Βρέθηκε: {tuple(target_y.shape)}"
        )

    if not isinstance(baseline_reference, torch.Tensor) or tuple(baseline_reference.shape) != (n_timestamps, n_nodes):
        raise ValueError(
            f"Το baseline_reference του split '{split_name}' δεν έχει αναμενόμενο shape "
            f"({n_timestamps}, {n_nodes}). Βρέθηκε: {tuple(baseline_reference.shape)}"
        )

    if not isinstance(observed_mask, torch.Tensor) or tuple(observed_mask.shape) != (n_timestamps, n_nodes):
        raise ValueError(
            f"Το observed_mask του split '{split_name}' δεν έχει αναμενόμενο shape "
            f"({n_timestamps}, {n_nodes}). Βρέθηκε: {tuple(observed_mask.shape)}"
        )

    if len(payload["node_ids"]) != n_nodes:
        raise ValueError(
            f"Ασυμφωνία node_ids length στο split '{split_name}': "
            f"{len(payload['node_ids'])} vs n_nodes={n_nodes}"
        )

    if len(payload["timestamps"]) != n_timestamps:
        raise ValueError(
            f"Ασυμφωνία timestamps length στο split '{split_name}': "
            f"{len(payload['timestamps'])} vs n_timestamps={n_timestamps}"
        )

    expected_n_nodes = int(nb12_run_config["n_nodes"])
    if n_nodes != expected_n_nodes:
        raise ValueError(
            f"Ασυμφωνία n_nodes στο split '{split_name}': "
            f"{n_nodes} vs NB12 expected {expected_n_nodes}"
        )


class OnTheFlyRefinedSnapshotDataset(Dataset):
    """
    Χτίζει snapshot-level PyG Data objects on the fly από το packaged dict payload,
    αλλάζοντας μόνο το runtime edge_index.
    """

    def __init__(self, payload: dict, refined_edge_index: np.ndarray):
        self.payload = payload
        self.refined_edge_index = torch.as_tensor(refined_edge_index, dtype=torch.long)

        self.static_x = payload["static_x"].float()
        self.dynamic_x = payload["dynamic_x"].float()
        self.target_y = payload["target_y"].float()
        self.baseline_reference = payload["baseline_reference"].float()
        self.observed_mask = payload["observed_mask"].bool()

        self.edge_attr_km = payload.get("edge_attr_km", None)
        if self.edge_attr_km is not None:
            self.edge_attr_km = self.edge_attr_km.float()

        self.n_timestamps = int(payload["n_timestamps"])

    def __len__(self):
        return self.n_timestamps

    def __getitem__(self, idx):
        x_t = torch.cat(
            [self.static_x, self.dynamic_x[idx]],
            dim=1,
        )

        y_t = self.target_y[idx].unsqueeze(-1)
        baseline_t = self.baseline_reference[idx].unsqueeze(-1)
        observed_mask_t = self.observed_mask[idx]

        data = Data(
            x=x_t,
            y=y_t,
            edge_index=self.refined_edge_index,
            observed_mask=observed_mask_t,
            baseline_reference=baseline_t,
            snapshot_idx=torch.tensor(idx, dtype=torch.long),
        )

        # Κρατάμε edge_attr μόνο αν το μέγεθος ταιριάζει ακριβώς.
        if self.edge_attr_km is not None and self.edge_attr_km.shape[0] == self.refined_edge_index.shape[1]:
            data.edge_attr_km = self.edge_attr_km

        return data


# ------------------------------------------------------------------
# Load packaged NB11 payloads
# ------------------------------------------------------------------
train_payload = torch_load_packaged_payload(project_config.NB11_TRAIN_GRAPH_DATASET, "train")
val_payload = torch_load_packaged_payload(project_config.NB11_VAL_GRAPH_DATASET, "validation")
test_payload = torch_load_packaged_payload(project_config.NB11_TEST_GRAPH_DATASET, "test")

validate_packaged_payload(train_payload, "train")
validate_packaged_payload(val_payload, "validation")
validate_packaged_payload(test_payload, "test")

# ------------------------------------------------------------------
# Derive runtime feature dimensions from actual packaged payloads
# ------------------------------------------------------------------
train_runtime_input_dim = int(train_payload["n_static_features"]) + int(train_payload["n_dynamic_features"])
val_runtime_input_dim = int(val_payload["n_static_features"]) + int(val_payload["n_dynamic_features"])
test_runtime_input_dim = int(test_payload["n_static_features"]) + int(test_payload["n_dynamic_features"])

if not (train_runtime_input_dim == val_runtime_input_dim == test_runtime_input_dim):
    raise ValueError(
        "Ασυμφωνία runtime input feature dimensions across splits: "
        f"train={train_runtime_input_dim}, val={val_runtime_input_dim}, test={test_runtime_input_dim}"
    )

NB14_RUNTIME_INPUT_FEATURE_DIM = train_runtime_input_dim

nb12_reference_input_dim = int(nb12_run_config["input_feature_dim"])
nb12_input_dim_matches_runtime = (NB14_RUNTIME_INPUT_FEATURE_DIM == nb12_reference_input_dim)

# ------------------------------------------------------------------
# Cross-check snapshot counts against NB12 reference config
# ------------------------------------------------------------------
expected_train_snapshots = int(nb12_run_config["n_train_snapshots"])
expected_val_snapshots = int(nb12_run_config["n_val_snapshots"])
expected_test_snapshots = int(nb12_run_config["n_test_snapshots"])

if int(train_payload["n_timestamps"]) != expected_train_snapshots:
    raise ValueError(
        f"Ασυμφωνία train snapshots: {train_payload['n_timestamps']} vs expected {expected_train_snapshots}"
    )

if int(val_payload["n_timestamps"]) != expected_val_snapshots:
    raise ValueError(
        f"Ασυμφωνία validation snapshots: {val_payload['n_timestamps']} vs expected {expected_val_snapshots}"
    )

if int(test_payload["n_timestamps"]) != expected_test_snapshots:
    raise ValueError(
        f"Ασυμφωνία test snapshots: {test_payload['n_timestamps']} vs expected {expected_test_snapshots}"
    )

# ------------------------------------------------------------------
# Build runtime datasets / loaders
# ------------------------------------------------------------------
nb14_batch_size = int(nb12_run_config["training_config"]["batch_size"])

NB14_RUNTIME_DATASETS = {}
NB14_RUNTIME_LOADERS = {}

for row in NB14_EXPERIMENT_REGISTRY.itertuples(index=False):
    refined_edge_index = NB14_EDGE_BUNDLES[row.experiment_id]["pruned_edge_index"]

    train_runtime_dataset = OnTheFlyRefinedSnapshotDataset(train_payload, refined_edge_index)
    val_runtime_dataset = OnTheFlyRefinedSnapshotDataset(val_payload, refined_edge_index)
    test_runtime_dataset = OnTheFlyRefinedSnapshotDataset(test_payload, refined_edge_index)

    NB14_RUNTIME_DATASETS[row.experiment_id] = {
        "train": train_runtime_dataset,
        "validation": val_runtime_dataset,
        "test": test_runtime_dataset,
    }

    NB14_RUNTIME_LOADERS[row.experiment_id] = {
        "train": DataLoader(train_runtime_dataset, batch_size=nb14_batch_size, shuffle=True),
        "validation": DataLoader(val_runtime_dataset, batch_size=nb14_batch_size, shuffle=False),
        "test": DataLoader(test_runtime_dataset, batch_size=nb14_batch_size, shuffle=False),
    }

# ------------------------------------------------------------------
# Inspect one runtime sample
# ------------------------------------------------------------------
reference_experiment_id = NB14_EXPERIMENT_REGISTRY.iloc[0]["experiment_id"]
reference_runtime_sample = NB14_RUNTIME_DATASETS[reference_experiment_id]["train"][0]

nb14_runtime_summary_df = pd.DataFrame(
    [
        {
            "experiment_id": experiment_id,
            "train_snapshots": len(runtime["train"]),
            "validation_snapshots": len(runtime["validation"]),
            "test_snapshots": len(runtime["test"]),
            "batch_size": nb14_batch_size,
            "runtime_input_feature_dim": NB14_RUNTIME_INPUT_FEATURE_DIM,
            "refined_directed_edges": NB14_EDGE_BUNDLES[experiment_id]["n_kept_directed_edges"],
        }
        for experiment_id, runtime in NB14_RUNTIME_DATASETS.items()
    ]
).sort_values(
    by=["experiment_id"],
    ascending=[True],
    kind="mergesort",
).reset_index(drop=True)

print("Packaged payload summary:")
print(f"- train static_x shape: {tuple(train_payload['static_x'].shape)}")
print(f"- train dynamic_x shape: {tuple(train_payload['dynamic_x'].shape)}")
print(f"- train target_y shape: {tuple(train_payload['target_y'].shape)}")
print(f"- train observed_mask shape: {tuple(train_payload['observed_mask'].shape)}")

print("\nRuntime input-dim note:")
print(f"- payload-derived runtime input_feature_dim: {NB14_RUNTIME_INPUT_FEATURE_DIM}")
print(f"- NB12 reference input_feature_dim: {nb12_reference_input_dim}")
print(f"- exact match with NB12 reference: {nb12_input_dim_matches_runtime}")

print("\nReference runtime sample contract:")
print(f"- x shape: {tuple(reference_runtime_sample.x.shape)}")
print(f"- y shape: {tuple(reference_runtime_sample.y.shape)}")
print(f"- edge_index shape: {tuple(reference_runtime_sample.edge_index.shape)}")
print(f"- observed_mask shape: {tuple(reference_runtime_sample.observed_mask.shape)}")
print(f"- baseline_reference shape: {tuple(reference_runtime_sample.baseline_reference.shape)}")

print("\nNB14 runtime dataset / loader summary:")
nb14_runtime_summary_df

Packaged payload summary:
- train static_x shape: (256, 5)
- train dynamic_x shape: (7819, 256, 36)
- train target_y shape: (7819, 256)
- train observed_mask shape: (7819, 256)

Runtime input-dim note:
- payload-derived runtime input_feature_dim: 41
- NB12 reference input_feature_dim: 42
- exact match with NB12 reference: False

Reference runtime sample contract:
- x shape: (256, 41)
- y shape: (256, 1)
- edge_index shape: (2, 534)
- observed_mask shape: (256,)
- baseline_reference shape: (256, 1)

NB14 runtime dataset / loader summary:


,experiment_id,train_snapshots,validation_snapshots,test_snapshots,batch_size,runtime_input_feature_dim,refined_directed_edges
0,NB14_E01,7819,720,4295,16,41,534
1,NB14_E02,7819,720,4295,16,41,588
2,NB14_E03,7819,720,4295,16,41,642


## Runtime feature-space note

Κατά το runtime materialization του `NB14`, το actual packaged graph payload του `NB11` έδωσε:

- `n_static_features = 5`
- `n_dynamic_features = 36`
- άρα **runtime input feature dimension = 41**

Την ίδια στιγμή, το compact `NB12` reference config περιέχει scalar τιμή:

- `input_feature_dim = 42`

### Πώς ερμηνεύεται αυτή η απόκλιση

Για το `NB14`, η **runtime authority** είναι το actual packaged payload schema και όχι μια μεμονωμένη scalar reference τιμή από παλαιότερο compact config export.

Άρα η απόκλιση `41 vs 42` αντιμετωπίζεται εδώ ως:

- **reference-metadata mismatch**
- και όχι ως corruption του packaged graph payload

### Γιατί αυτό δεν σπάει το benchmark contract

Το `NB14` δεν αλλάζει το feature space.  
Δεν αφαιρεί και δεν προσθέτει features ad hoc.

Αντίθετα:

- χρησιμοποιεί το ήδη packaged feature space όπως έχει serialized από το `NB11`
- κρατά σταθερά τα split counts, το node count και το topology-only refinement scope
- και συνεχίζει με frozen model-family logic πάνω στο actual runtime contract

### Μεθοδολογική συνέπεια

Στο υπόλοιπο notebook, οποιοδήποτε model-definition ή forward pass θα πρέπει να χρησιμοποιεί ως input dimension το:

`NB14_RUNTIME_INPUT_FEATURE_DIM`

και όχι να επιβάλλει μηχανικά το scalar `NB12` reference field.

In [11]:
# NB14 | Frozen model family and masked evaluation helpers

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch_geometric.nn import GCNConv


# ------------------------------------------------------------------
# Frozen training/model-family hyperparameters from NB12 reference
# ------------------------------------------------------------------
NB14_HIDDEN_CHANNELS = int(nb12_run_config["training_config"]["hidden_channels"])
NB14_DROPOUT = float(nb12_run_config["training_config"]["dropout"])
NB14_LEARNING_RATE = float(nb12_run_config["training_config"]["learning_rate"])
NB14_WEIGHT_DECAY = float(nb12_run_config["training_config"]["weight_decay"])
NB14_MAX_EPOCHS = int(nb12_run_config["training_config"]["max_epochs"])
NB14_PATIENCE = int(nb12_run_config["training_config"]["patience"])
NB14_MIN_DELTA = float(nb12_run_config["training_config"]["min_delta"])
NB14_GRADIENT_CLIP_NORM = float(nb12_run_config["training_config"]["gradient_clip_norm"])


class NB14FrozenGCN(torch.nn.Module):
    """
    Frozen GCN family for NB14:
    - ίδια γενική model family με NB12/NB13
    - topology refinement μόνο μέσω edge_index
    - input dim από το actual packaged runtime payload
    """

    def __init__(
        self,
        input_dim: int,
        hidden_channels: int,
        message_passing_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()

        if message_passing_layers < 1:
            raise ValueError("Το message_passing_layers πρέπει να είναι >= 1.")

        self.input_dim = int(input_dim)
        self.hidden_channels = int(hidden_channels)
        self.message_passing_layers = int(message_passing_layers)
        self.dropout = float(dropout)

        self.input_conv = GCNConv(self.input_dim, self.hidden_channels)

        self.hidden_convs = torch.nn.ModuleList(
            [
                GCNConv(self.hidden_channels, self.hidden_channels)
                for _ in range(self.message_passing_layers - 1)
            ]
        )

        self.output_head = torch.nn.Linear(self.hidden_channels, 1)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index

        x = self.input_conv(x, edge_index)
        x = torch.relu(x)
        x = torch.nn.functional.dropout(
            x,
            p=self.dropout,
            training=self.training,
        )

        for conv in self.hidden_convs:
            x = conv(x, edge_index)
            x = torch.relu(x)
            x = torch.nn.functional.dropout(
                x,
                p=self.dropout,
                training=self.training,
            )

        x = self.output_head(x)
        return x.squeeze(-1)


def extract_masked_targets_and_predictions(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    observed_mask: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Επιστρέφει masked prediction / target tensors σε flattened μορφή.
    """
    preds = predictions.reshape(-1)
    targs = targets.reshape(-1)
    mask = observed_mask.reshape(-1).bool()

    if preds.shape != targs.shape:
        raise ValueError(
            f"Ασυμφωνία prediction/target shapes: {preds.shape} vs {targs.shape}"
        )

    if preds.shape != mask.shape:
        raise ValueError(
            f"Ασυμφωνία prediction/mask shapes: {preds.shape} vs {mask.shape}"
        )

    if mask.sum().item() == 0:
        raise ValueError("Το observed_mask δεν περιέχει ούτε ένα observed στοιχείο.")

    return preds[mask], targs[mask]


def masked_mse_loss(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    observed_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Primary optimization loss:
    masked MSE πάνω στο observed prediction space.
    """
    masked_preds, masked_targs = extract_masked_targets_and_predictions(
        predictions,
        targets,
        observed_mask,
    )
    return torch.mean((masked_preds - masked_targs) ** 2)


def compute_masked_regression_metrics(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    observed_mask: torch.Tensor,
) -> dict:
    """
    Υπολογίζει MAE / RMSE / R² μόνο πάνω στο observed prediction space.
    """
    masked_preds, masked_targs = extract_masked_targets_and_predictions(
        predictions.detach().cpu(),
        targets.detach().cpu(),
        observed_mask.detach().cpu(),
    )

    y_pred = masked_preds.numpy()
    y_true = masked_targs.numpy()

    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))

    # Το r2_score χρειάζεται τουλάχιστον 2 σημεία.
    if len(y_true) < 2:
        r2 = float("nan")
    else:
        r2 = float(r2_score(y_true, y_pred))

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "n_observed_points": int(len(y_true)),
    }


@torch.no_grad()
def run_model_on_batch(model: torch.nn.Module, batch, device: torch.device) -> torch.Tensor:
    """
    Βοηθητικό dry-run / evaluation helper για ένα batch.
    """
    batch = batch.to(device)
    predictions = model(batch)
    return predictions


# ------------------------------------------------------------------
# Dry-run forward pass on one reference batch
# ------------------------------------------------------------------
reference_experiment_id = NB14_EXPERIMENT_REGISTRY.iloc[0]["experiment_id"]
reference_message_passing_layers = int(
    NB14_EXPERIMENT_REGISTRY.loc[
        NB14_EXPERIMENT_REGISTRY["experiment_id"] == reference_experiment_id,
        "message_passing_layers",
    ].iloc[0]
)

nb14_reference_model = NB14FrozenGCN(
    input_dim=NB14_RUNTIME_INPUT_FEATURE_DIM,
    hidden_channels=NB14_HIDDEN_CHANNELS,
    message_passing_layers=reference_message_passing_layers,
    dropout=NB14_DROPOUT,
).to(DEVICE)

reference_train_loader = NB14_RUNTIME_LOADERS[reference_experiment_id]["train"]
reference_batch = next(iter(reference_train_loader))
reference_batch = reference_batch.to(DEVICE)

reference_predictions = nb14_reference_model(reference_batch)
reference_loss = masked_mse_loss(
    predictions=reference_predictions,
    targets=reference_batch.y.squeeze(-1),
    observed_mask=reference_batch.observed_mask,
)

print("NB14 frozen model-family summary:")
print(f"- runtime input_dim: {NB14_RUNTIME_INPUT_FEATURE_DIM}")
print(f"- hidden_channels: {NB14_HIDDEN_CHANNELS}")
print(f"- dropout: {NB14_DROPOUT}")
print(f"- reference message_passing_layers: {reference_message_passing_layers}")
print(f"- learning_rate: {NB14_LEARNING_RATE}")
print(f"- weight_decay: {NB14_WEIGHT_DECAY}")
print(f"- max_epochs: {NB14_MAX_EPOCHS}")
print(f"- patience: {NB14_PATIENCE}")
print(f"- min_delta: {NB14_MIN_DELTA}")
print(f"- gradient_clip_norm: {NB14_GRADIENT_CLIP_NORM}")

print("\nReference batch dry-run:")
print(f"- batch x shape: {tuple(reference_batch.x.shape)}")
print(f"- batch y shape: {tuple(reference_batch.y.shape)}")
print(f"- batch edge_index shape: {tuple(reference_batch.edge_index.shape)}")
print(f"- batch observed_mask shape: {tuple(reference_batch.observed_mask.shape)}")
print(f"- predictions shape: {tuple(reference_predictions.shape)}")
print(f"- masked MSE dry-run loss: {float(reference_loss.detach().cpu()):.6f}")

NB14 frozen model-family summary:
- runtime input_dim: 41
- hidden_channels: 64
- dropout: 0.2
- reference message_passing_layers: 2
- learning_rate: 0.001
- weight_decay: 1e-05
- max_epochs: 30
- patience: 6
- min_delta: 1e-05
- gradient_clip_norm: 1.0

Reference batch dry-run:
- batch x shape: (4096, 41)
- batch y shape: (4096, 1)
- batch edge_index shape: (2, 8544)
- batch observed_mask shape: (4096,)
- predictions shape: (4096,)
- masked MSE dry-run loss: nan


In [12]:
# NB14 | Finite-value diagnostic inspection before training
# Δεν προχωράμε σε training αν το dry-run numerical contract δεν είναι finite.

def tensor_finite_summary(name: str, tensor: torch.Tensor) -> dict:
    tensor_cpu = tensor.detach().cpu()
    is_nan = torch.isnan(tensor_cpu)
    is_inf = torch.isinf(tensor_cpu)
    is_finite = torch.isfinite(tensor_cpu)

    return {
        "name": name,
        "shape": tuple(tensor_cpu.shape),
        "numel": int(tensor_cpu.numel()),
        "n_nan": int(is_nan.sum().item()),
        "n_inf": int(is_inf.sum().item()),
        "n_finite": int(is_finite.sum().item()),
        "finite_ratio": float(is_finite.float().mean().item()),
    }


def print_summary(summary: dict) -> None:
    print(f"{summary['name']}:")
    print(f"  - shape: {summary['shape']}")
    print(f"  - numel: {summary['numel']}")
    print(f"  - n_nan: {summary['n_nan']}")
    print(f"  - n_inf: {summary['n_inf']}")
    print(f"  - n_finite: {summary['n_finite']}")
    print(f"  - finite_ratio: {summary['finite_ratio']:.6f}")


with torch.no_grad():
    diagnostic_predictions = nb14_reference_model(reference_batch)

diagnostic_targets = reference_batch.y.squeeze(-1)
diagnostic_mask = reference_batch.observed_mask.bool()

print("Observed-space diagnostic:")
print(f"- observed points in batch: {int(diagnostic_mask.sum().item())}")
print(f"- total points in batch: {int(diagnostic_mask.numel())}")

print("\nTensor finite summaries:")
for summary in [
    tensor_finite_summary("reference_batch.x", reference_batch.x),
    tensor_finite_summary("reference_batch.y.squeeze(-1)", diagnostic_targets),
    tensor_finite_summary("reference_batch.observed_mask", diagnostic_mask.float()),
    tensor_finite_summary("diagnostic_predictions", diagnostic_predictions),
]:
    print_summary(summary)

# Masked extraction diagnostic
masked_preds, masked_targs = extract_masked_targets_and_predictions(
    predictions=diagnostic_predictions,
    targets=diagnostic_targets,
    observed_mask=diagnostic_mask,
)

print("\nMasked-space finite summaries:")
for summary in [
    tensor_finite_summary("masked_preds", masked_preds),
    tensor_finite_summary("masked_targs", masked_targs),
]:
    print_summary(summary)

# Feature-wise NaN inspection στο input x
x_nan_per_feature = torch.isnan(reference_batch.x).sum(dim=0).detach().cpu()
x_inf_per_feature = torch.isinf(reference_batch.x).sum(dim=0).detach().cpu()

feature_issue_df = pd.DataFrame(
    {
        "feature_index": np.arange(reference_batch.x.shape[1]),
        "n_nan": x_nan_per_feature.numpy(),
        "n_inf": x_inf_per_feature.numpy(),
    }
)

feature_issue_df["n_non_finite"] = feature_issue_df["n_nan"] + feature_issue_df["n_inf"]
feature_issue_df = feature_issue_df.sort_values(
    by=["n_non_finite", "feature_index"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)

print("\nTop feature-level non-finite issues in x:")
display(feature_issue_df.head(10))

# Final dry-run status flag
all_core_finite = (
    torch.isfinite(reference_batch.x).all().item()
    and torch.isfinite(diagnostic_targets).all().item()
    and torch.isfinite(diagnostic_predictions).all().item()
)

print("\nDry-run finite status:")
print(f"- all core tensors finite: {bool(all_core_finite)}")

Observed-space diagnostic:
- observed points in batch: 4055
- total points in batch: 4096

Tensor finite summaries:
reference_batch.x:
  - shape: (4096, 41)
  - numel: 167936
  - n_nan: 1476
  - n_inf: 0
  - n_finite: 166460
  - finite_ratio: 0.991211
reference_batch.y.squeeze(-1):
  - shape: (4096,)
  - numel: 4096
  - n_nan: 41
  - n_inf: 0
  - n_finite: 4055
  - finite_ratio: 0.989990
reference_batch.observed_mask:
  - shape: (4096,)
  - numel: 4096
  - n_nan: 0
  - n_inf: 0
  - n_finite: 4096
  - finite_ratio: 1.000000
diagnostic_predictions:
  - shape: (4096,)
  - numel: 4096
  - n_nan: 196
  - n_inf: 0
  - n_finite: 3900
  - finite_ratio: 0.952148

Masked-space finite summaries:
masked_preds:
  - shape: (4055,)
  - numel: 4055
  - n_nan: 155
  - n_inf: 0
  - n_finite: 3900
  - finite_ratio: 0.961776
masked_targs:
  - shape: (4055,)
  - numel: 4055
  - n_nan: 0
  - n_inf: 0
  - n_finite: 4055
  - finite_ratio: 1.000000

Top feature-level non-finite issues in x:


,feature_index,n_nan,n_inf,n_non_finite
0,5,41,0,41
1,6,41,0,41
2,7,41,0,41
3,8,41,0,41
4,9,41,0,41
5,10,41,0,41
6,11,41,0,41
7,12,41,0,41
8,13,41,0,41
9,14,41,0,41



Dry-run finite status:
- all core tensors finite: False


In [13]:
# NB14 | Row-level alignment check for non-finite x vs masked-out targets

row_has_non_finite_x = ~torch.isfinite(reference_batch.x).all(dim=1)
row_has_nan_y = torch.isnan(reference_batch.y.squeeze(-1))
row_is_unobserved = ~reference_batch.observed_mask.bool()

alignment_df = pd.DataFrame(
    {
        "row_idx": np.arange(reference_batch.x.shape[0]),
        "has_non_finite_x": row_has_non_finite_x.detach().cpu().numpy(),
        "has_nan_y": row_has_nan_y.detach().cpu().numpy(),
        "is_unobserved": row_is_unobserved.detach().cpu().numpy(),
    }
)

alignment_df["all_three_match"] = (
    alignment_df["has_non_finite_x"]
    == alignment_df["has_nan_y"]
) & (
    alignment_df["has_nan_y"]
    == alignment_df["is_unobserved"]
)

print("Row-level alignment summary:")
print(f"- rows with non-finite x: {int(alignment_df['has_non_finite_x'].sum())}")
print(f"- rows with NaN y: {int(alignment_df['has_nan_y'].sum())}")
print(f"- rows with unobserved mask: {int(alignment_df['is_unobserved'].sum())}")
print(f"- rows where all three indicators match: {int(alignment_df['all_three_match'].sum())} / {len(alignment_df)}")

exact_alignment = (
    alignment_df["has_non_finite_x"].equals(alignment_df["has_nan_y"])
    and alignment_df["has_nan_y"].equals(alignment_df["is_unobserved"])
)

print(f"- exact full alignment: {exact_alignment}")

print("\nRows with any issue:")
display(alignment_df.loc[
    alignment_df["has_non_finite_x"]
    | alignment_df["has_nan_y"]
    | alignment_df["is_unobserved"]
].head(20))

Row-level alignment summary:
- rows with non-finite x: 41
- rows with NaN y: 41
- rows with unobserved mask: 41
- rows where all three indicators match: 4096 / 4096
- exact full alignment: True

Rows with any issue:


,row_idx,has_non_finite_x,has_nan_y,is_unobserved,all_three_match
178,178,True,True,True,True
205,205,True,True,True,True
434,434,True,True,True,True
461,461,True,True,True,True
531,531,True,True,True,True
662,662,True,True,True,True
934,934,True,True,True,True
1051,1051,True,True,True,True
1299,1299,True,True,True,True
1430,1430,True,True,True,True


In [14]:
# NB14 | Runtime stabilization patch for masked-out / unobserved rows
# Εφόσον αποδείχθηκε exact alignment:
# non-finite x <-> NaN y <-> observed_mask == False,
# κάνουμε zero-fill μόνο στο unobserved runtime space.

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


class StabilizedOnTheFlyRefinedSnapshotDataset(Dataset):
    """
    Snapshot-level runtime dataset με topology refinement και numerical stabilization.
    Η σταθεροποίηση εφαρμόζεται μόνο στο unobserved space.
    """

    def __init__(self, payload: dict, refined_edge_index: np.ndarray):
        self.payload = payload
        self.refined_edge_index = torch.as_tensor(refined_edge_index, dtype=torch.long)

        self.static_x = payload["static_x"].float()
        self.dynamic_x = payload["dynamic_x"].float()
        self.target_y = payload["target_y"].float()
        self.baseline_reference = payload["baseline_reference"].float()
        self.observed_mask = payload["observed_mask"].bool()

        self.edge_attr_km = payload.get("edge_attr_km", None)
        if self.edge_attr_km is not None:
            self.edge_attr_km = self.edge_attr_km.float()

        self.n_timestamps = int(payload["n_timestamps"])

    def __len__(self):
        return self.n_timestamps

    def __getitem__(self, idx):
        x_t = torch.cat(
            [self.static_x, self.dynamic_x[idx]],
            dim=1,
        ).clone()

        y_t = self.target_y[idx].unsqueeze(-1).clone()
        baseline_t = self.baseline_reference[idx].unsqueeze(-1).clone()
        observed_mask_t = self.observed_mask[idx].clone().bool()

        unobserved_rows = ~observed_mask_t

        # Zero-fill μόνο στο masked-out / unobserved space.
        if unobserved_rows.any():
            x_t[unobserved_rows] = 0.0
            y_t[unobserved_rows] = 0.0
            baseline_t[unobserved_rows] = 0.0

        # Extra numerical guard για residual non-finite values.
        x_t = torch.nan_to_num(x_t, nan=0.0, posinf=0.0, neginf=0.0)
        y_t = torch.nan_to_num(y_t, nan=0.0, posinf=0.0, neginf=0.0)
        baseline_t = torch.nan_to_num(baseline_t, nan=0.0, posinf=0.0, neginf=0.0)

        data = Data(
            x=x_t,
            y=y_t,
            edge_index=self.refined_edge_index,
            observed_mask=observed_mask_t,
            baseline_reference=baseline_t,
            snapshot_idx=torch.tensor(idx, dtype=torch.long),
        )

        # Κρατάμε edge_attr μόνο αν ταιριάζει ακριβώς με το refined topology size.
        if self.edge_attr_km is not None and self.edge_attr_km.shape[0] == self.refined_edge_index.shape[1]:
            data.edge_attr_km = self.edge_attr_km

        return data


# ------------------------------------------------------------------
# Rebuild runtime datasets / loaders with stabilization
# ------------------------------------------------------------------
NB14_RUNTIME_DATASETS = {}
NB14_RUNTIME_LOADERS = {}

for row in NB14_EXPERIMENT_REGISTRY.itertuples(index=False):
    refined_edge_index = NB14_EDGE_BUNDLES[row.experiment_id]["pruned_edge_index"]

    train_runtime_dataset = StabilizedOnTheFlyRefinedSnapshotDataset(train_payload, refined_edge_index)
    val_runtime_dataset = StabilizedOnTheFlyRefinedSnapshotDataset(val_payload, refined_edge_index)
    test_runtime_dataset = StabilizedOnTheFlyRefinedSnapshotDataset(test_payload, refined_edge_index)

    NB14_RUNTIME_DATASETS[row.experiment_id] = {
        "train": train_runtime_dataset,
        "validation": val_runtime_dataset,
        "test": test_runtime_dataset,
    }

    NB14_RUNTIME_LOADERS[row.experiment_id] = {
        "train": DataLoader(train_runtime_dataset, batch_size=nb14_batch_size, shuffle=True),
        "validation": DataLoader(val_runtime_dataset, batch_size=nb14_batch_size, shuffle=False),
        "test": DataLoader(test_runtime_dataset, batch_size=nb14_batch_size, shuffle=False),
    }

NB14_RUNTIME_STABILIZATION_APPLIED = True

# ------------------------------------------------------------------
# Re-run dry-run forward check after stabilization
# ------------------------------------------------------------------
reference_experiment_id = NB14_EXPERIMENT_REGISTRY.iloc[0]["experiment_id"]
reference_message_passing_layers = int(
    NB14_EXPERIMENT_REGISTRY.loc[
        NB14_EXPERIMENT_REGISTRY["experiment_id"] == reference_experiment_id,
        "message_passing_layers",
    ].iloc[0]
)

nb14_reference_model_stabilized = NB14FrozenGCN(
    input_dim=NB14_RUNTIME_INPUT_FEATURE_DIM,
    hidden_channels=NB14_HIDDEN_CHANNELS,
    message_passing_layers=reference_message_passing_layers,
    dropout=NB14_DROPOUT,
).to(DEVICE)

reference_train_loader = NB14_RUNTIME_LOADERS[reference_experiment_id]["train"]
reference_batch_stabilized = next(iter(reference_train_loader)).to(DEVICE)

reference_predictions_stabilized = nb14_reference_model_stabilized(reference_batch_stabilized)
reference_loss_stabilized = masked_mse_loss(
    predictions=reference_predictions_stabilized,
    targets=reference_batch_stabilized.y.squeeze(-1),
    observed_mask=reference_batch_stabilized.observed_mask,
)

stabilized_core_finite = (
    torch.isfinite(reference_batch_stabilized.x).all().item()
    and torch.isfinite(reference_batch_stabilized.y).all().item()
    and torch.isfinite(reference_predictions_stabilized).all().item()
    and torch.isfinite(reference_loss_stabilized).all().item()
)

print("NB14 runtime stabilization summary:")
print(f"- stabilization applied: {NB14_RUNTIME_STABILIZATION_APPLIED}")
print(f"- reference experiment_id: {reference_experiment_id}")
print(f"- stabilized batch x shape: {tuple(reference_batch_stabilized.x.shape)}")
print(f"- stabilized batch y shape: {tuple(reference_batch_stabilized.y.shape)}")
print(f"- stabilized batch edge_index shape: {tuple(reference_batch_stabilized.edge_index.shape)}")
print(f"- stabilized batch observed_mask shape: {tuple(reference_batch_stabilized.observed_mask.shape)}")
print(f"- stabilized predictions shape: {tuple(reference_predictions_stabilized.shape)}")
print(f"- stabilized masked MSE dry-run loss: {float(reference_loss_stabilized.detach().cpu()):.6f}")
print(f"- all stabilized core tensors finite: {bool(stabilized_core_finite)}")

NB14 runtime stabilization summary:
- stabilization applied: True
- reference experiment_id: NB14_E01
- stabilized batch x shape: (4096, 41)
- stabilized batch y shape: (4096, 1)
- stabilized batch edge_index shape: (2, 8544)
- stabilized batch observed_mask shape: (4096,)
- stabilized predictions shape: (4096,)
- stabilized masked MSE dry-run loss: 18566946.000000
- all stabilized core tensors finite: True


In [15]:
# NB14 | Validation-only training and model selection
# Εδώ ΔΕΝ αγγίζουμε το test split για selection.

import copy
import time


def clone_state_dict_to_cpu(state_dict: dict) -> dict:
    """
    Κρατά καθαρό CPU copy του best checkpoint για notebook-safe reuse.
    """
    return {k: v.detach().cpu().clone() for k, v in state_dict.items()}


def resolve_optimizer(model: torch.nn.Module):
    """
    Επιλύει optimizer με ήπιο fallback στο Adam αν δεν υπάρχει explicit field.
    """
    optimizer_name = str(
        nb12_run_config["training_config"].get(
            "optimizer_name",
            nb12_run_config["training_config"].get("optimizer", "Adam"),
        )
    ).lower()

    if optimizer_name == "adamw":
        return torch.optim.AdamW(
            model.parameters(),
            lr=NB14_LEARNING_RATE,
            weight_decay=NB14_WEIGHT_DECAY,
        )

    # Default-safe fallback
    return torch.optim.Adam(
        model.parameters(),
        lr=NB14_LEARNING_RATE,
        weight_decay=NB14_WEIGHT_DECAY,
    )


def run_train_epoch(
    model: torch.nn.Module,
    loader,
    optimizer,
    device: torch.device,
    gradient_clip_norm: float,
) -> float:
    """
    Ένα training epoch πάνω στο observed prediction space.
    """
    model.train()

    batch_losses = []

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(batch)
        loss = masked_mse_loss(
            predictions=predictions,
            targets=batch.y.squeeze(-1),
            observed_mask=batch.observed_mask,
        )

        if not torch.isfinite(loss):
            raise ValueError("Βρέθηκε μη finite training loss κατά το train epoch.")

        loss.backward()

        if gradient_clip_norm is not None and gradient_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)

        optimizer.step()
        batch_losses.append(float(loss.detach().cpu()))

    if len(batch_losses) == 0:
        raise ValueError("Το training loader δεν παρήγαγε batches.")

    return float(np.mean(batch_losses))


@torch.no_grad()
def evaluate_loader(
    model: torch.nn.Module,
    loader,
    device: torch.device,
) -> dict:
    """
    Evaluation μόνο στο observed prediction space.
    Επιστρέφει aggregate loss + MAE/RMSE/R².
    """
    model.eval()

    batch_losses = []
    all_masked_preds = []
    all_masked_targs = []

    for batch in loader:
        batch = batch.to(device)

        predictions = model(batch)
        loss = masked_mse_loss(
            predictions=predictions,
            targets=batch.y.squeeze(-1),
            observed_mask=batch.observed_mask,
        )

        if not torch.isfinite(loss):
            raise ValueError("Βρέθηκε μη finite evaluation loss.")

        masked_preds, masked_targs = extract_masked_targets_and_predictions(
            predictions=predictions.detach().cpu(),
            targets=batch.y.squeeze(-1).detach().cpu(),
            observed_mask=batch.observed_mask.detach().cpu(),
        )

        all_masked_preds.append(masked_preds)
        all_masked_targs.append(masked_targs)
        batch_losses.append(float(loss.detach().cpu()))

    if len(batch_losses) == 0:
        raise ValueError("Το evaluation loader δεν παρήγαγε batches.")

    y_pred = torch.cat(all_masked_preds, dim=0)
    y_true = torch.cat(all_masked_targs, dim=0)

    metrics = compute_masked_regression_metrics(
        predictions=y_pred,
        targets=y_true,
        observed_mask=torch.ones_like(y_true, dtype=torch.bool),
    )
    metrics["loss"] = float(np.mean(batch_losses))
    return metrics


NB14_EXPERIMENT_RESULTS = {}
validation_rows = []

print("NB14 validation-only training started...")

for row in NB14_EXPERIMENT_REGISTRY.itertuples(index=False):
    experiment_id = str(row.experiment_id)
    message_passing_layers = int(row.message_passing_layers)

    # Re-seed για fairer experiment-to-experiment comparison.
    seed_everything(SEED)

    model = NB14FrozenGCN(
        input_dim=NB14_RUNTIME_INPUT_FEATURE_DIM,
        hidden_channels=NB14_HIDDEN_CHANNELS,
        message_passing_layers=message_passing_layers,
        dropout=NB14_DROPOUT,
    ).to(DEVICE)

    optimizer = resolve_optimizer(model)

    train_loader = NB14_RUNTIME_LOADERS[experiment_id]["train"]
    val_loader = NB14_RUNTIME_LOADERS[experiment_id]["validation"]

    best_state_dict = None
    best_epoch = None
    best_val_loss = float("inf")
    best_val_metrics = None

    bad_epochs = 0
    history_rows = []

    experiment_start_time = time.perf_counter()

    for epoch in range(1, NB14_MAX_EPOCHS + 1):
        train_loss = run_train_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=DEVICE,
            gradient_clip_norm=NB14_GRADIENT_CLIP_NORM,
        )

        val_metrics = evaluate_loader(
            model=model,
            loader=val_loader,
            device=DEVICE,
        )

        history_rows.append(
            {
                "experiment_id": experiment_id,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_MAE": val_metrics["MAE"],
                "val_RMSE": val_metrics["RMSE"],
                "val_R2": val_metrics["R2"],
                "val_n_observed_points": val_metrics["n_observed_points"],
            }
        )

        improved = (best_val_loss - val_metrics["loss"]) > NB14_MIN_DELTA

        if improved:
            best_val_loss = float(val_metrics["loss"])
            best_epoch = int(epoch)
            best_val_metrics = dict(val_metrics)
            best_state_dict = clone_state_dict_to_cpu(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= NB14_PATIENCE:
            break

    if best_state_dict is None or best_epoch is None or best_val_metrics is None:
        raise RuntimeError(f"Δεν αποθηκεύτηκε valid best checkpoint για το {experiment_id}.")

    experiment_elapsed_sec = float(time.perf_counter() - experiment_start_time)
    history_df = pd.DataFrame(history_rows)

    stopped_early = len(history_df) < NB14_MAX_EPOCHS

    NB14_EXPERIMENT_RESULTS[experiment_id] = {
        "best_state_dict": best_state_dict,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_metrics": best_val_metrics,
        "history_df": history_df,
        "elapsed_seconds": experiment_elapsed_sec,
        "stopped_early": stopped_early,
    }

    validation_rows.append(
        {
            "experiment_id": experiment_id,
            "topology_variant": row.topology_variant,
            "edge_policy": row.edge_policy,
            "edge_keep_quantile": float(row.edge_keep_quantile),
            "message_passing_layers": message_passing_layers,
            "reference_nb13_experiment_id": row.reference_nb13_experiment_id,
            "is_nb13_reference_quantile": bool(row.is_nb13_reference_quantile),
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "best_val_MAE": float(best_val_metrics["MAE"]),
            "best_val_RMSE": float(best_val_metrics["RMSE"]),
            "best_val_R2": float(best_val_metrics["R2"]),
            "val_n_observed_points": int(best_val_metrics["n_observed_points"]),
            "epochs_completed": int(len(history_df)),
            "stopped_early": bool(stopped_early),
            "elapsed_seconds": experiment_elapsed_sec,
            "runtime_input_feature_dim": int(NB14_RUNTIME_INPUT_FEATURE_DIM),
            "refined_directed_edges": int(
                NB14_EDGE_BUNDLES[experiment_id]["n_kept_directed_edges"]
            ),
        }
    )

    print(
        f"[{experiment_id}] "
        f"best_epoch={best_epoch} | "
        f"best_val_loss={best_val_loss:.6f} | "
        f"best_val_MAE={best_val_metrics['MAE']:.6f} | "
        f"epochs={len(history_df)}"
    )

nb14_validation_summary_df = pd.DataFrame(validation_rows).sort_values(
    by=["best_val_loss", "best_val_MAE", "edge_keep_quantile", "experiment_id"],
    ascending=[True, True, True, True],
    kind="mergesort",
).reset_index(drop=True)

NB14_SELECTED_EXPERIMENT_ID = str(nb14_validation_summary_df.iloc[0]["experiment_id"])
NB14_SELECTED_RESULT = NB14_EXPERIMENT_RESULTS[NB14_SELECTED_EXPERIMENT_ID]
NB14_SELECTED_HISTORY_DF = NB14_SELECTED_RESULT["history_df"].copy()

nb14_validation_summary_df["is_selected_for_final_test"] = (
    nb14_validation_summary_df["experiment_id"] == NB14_SELECTED_EXPERIMENT_ID
)

print("\nNB14 validation-only selection result:")
print(f"- selected experiment_id: {NB14_SELECTED_EXPERIMENT_ID}")
print(f"- selected best_epoch: {NB14_SELECTED_RESULT['best_epoch']}")
print(f"- selected best_val_loss: {NB14_SELECTED_RESULT['best_val_loss']:.6f}")
print(f"- selected best_val_MAE: {NB14_SELECTED_RESULT['best_val_metrics']['MAE']:.6f}")

print("\nNB14 validation summary:")
nb14_validation_summary_df

NB14 validation-only training started...
[NB14_E01] best_epoch=5 | best_val_loss=0.060545 | best_val_MAE=0.179216 | epochs=11
[NB14_E02] best_epoch=5 | best_val_loss=0.060570 | best_val_MAE=0.179569 | epochs=11
[NB14_E03] best_epoch=5 | best_val_loss=0.060570 | best_val_MAE=0.179570 | epochs=11

NB14 validation-only selection result:
- selected experiment_id: NB14_E01
- selected best_epoch: 5
- selected best_val_loss: 0.060545
- selected best_val_MAE: 0.179216

NB14 validation summary:


,experiment_id,topology_variant,edge_policy,edge_keep_quantile,message_passing_layers,reference_nb13_experiment_id,is_nb13_reference_quantile,best_epoch,best_val_loss,best_val_MAE,best_val_RMSE,best_val_R2,val_n_observed_points,epochs_completed,stopped_early,elapsed_seconds,runtime_input_feature_dim,refined_directed_edges,is_selected_for_final_test
0,NB14_E01,local_pruned_graph,keep_shortest_edges_only,0.50,2,NB13_E06,True,5,0.060545,0.179216,0.245976,-0.005676,182998,11,True,157.289608,41,534,True
1,NB14_E02,local_pruned_graph,keep_shortest_edges_only,0.55,2,NB13_E06,False,5,0.060570,0.179569,0.246028,-0.006100,182998,11,True,158.218921,41,588,False
2,NB14_E03,local_pruned_graph,keep_shortest_edges_only,0.60,2,NB13_E06,False,5,0.060570,0.179570,0.246028,-0.006101,182998,11,True,165.750042,41,642,False


In [16]:
# NB14 | Test-only final reporting for the validation-selected experiment
# Εδώ αγγίζουμε το test split μόνο μία φορά και μόνο για το selected experiment.

# ------------------------------------------------------------------
# Resolve selected experiment configuration
# ------------------------------------------------------------------
selected_row = nb14_validation_summary_df.loc[
    nb14_validation_summary_df["experiment_id"] == NB14_SELECTED_EXPERIMENT_ID
].iloc[0]

selected_message_passing_layers = int(selected_row["message_passing_layers"])
selected_edge_keep_quantile = float(selected_row["edge_keep_quantile"])
selected_refined_directed_edges = int(selected_row["refined_directed_edges"])

# ------------------------------------------------------------------
# Rebuild fresh model instance and load best selected checkpoint
# ------------------------------------------------------------------
nb14_selected_model = NB14FrozenGCN(
    input_dim=NB14_RUNTIME_INPUT_FEATURE_DIM,
    hidden_channels=NB14_HIDDEN_CHANNELS,
    message_passing_layers=selected_message_passing_layers,
    dropout=NB14_DROPOUT,
).to(DEVICE)

nb14_selected_model.load_state_dict(NB14_SELECTED_RESULT["best_state_dict"])
nb14_selected_model.eval()

# ------------------------------------------------------------------
# Test-only evaluation
# ------------------------------------------------------------------
selected_test_loader = NB14_RUNTIME_LOADERS[NB14_SELECTED_EXPERIMENT_ID]["test"]

nb14_selected_test_metrics = evaluate_loader(
    model=nb14_selected_model,
    loader=selected_test_loader,
    device=DEVICE,
)

NB14_TEST_EVALUATION_COMPLETED = True
NB14_TEST_EVALUATED_EXPERIMENT_IDS = [NB14_SELECTED_EXPERIMENT_ID]

NB14_FINAL_TEST_RESULT = {
    "experiment_id": NB14_SELECTED_EXPERIMENT_ID,
    "topology_variant": str(selected_row["topology_variant"]),
    "edge_policy": str(selected_row["edge_policy"]),
    "edge_keep_quantile": selected_edge_keep_quantile,
    "message_passing_layers": selected_message_passing_layers,
    "best_epoch": int(NB14_SELECTED_RESULT["best_epoch"]),
    "best_val_loss": float(NB14_SELECTED_RESULT["best_val_loss"]),
    "best_val_MAE": float(NB14_SELECTED_RESULT["best_val_metrics"]["MAE"]),
    "test_loss": float(nb14_selected_test_metrics["loss"]),
    "test_MAE": float(nb14_selected_test_metrics["MAE"]),
    "test_RMSE": float(nb14_selected_test_metrics["RMSE"]),
    "test_R2": float(nb14_selected_test_metrics["R2"]),
    "test_n_observed_points": int(nb14_selected_test_metrics["n_observed_points"]),
    "runtime_input_feature_dim": int(NB14_RUNTIME_INPUT_FEATURE_DIM),
    "refined_directed_edges": selected_refined_directed_edges,
    "reference_nb13_experiment_id": str(selected_row["reference_nb13_experiment_id"]),
    "is_nb13_reference_quantile": bool(selected_row["is_nb13_reference_quantile"]),
}

nb14_test_summary_df = pd.DataFrame([NB14_FINAL_TEST_RESULT])

print("NB14 test-only final reporting:")
print(f"- selected experiment_id: {NB14_SELECTED_EXPERIMENT_ID}")
print(f"- selected message_passing_layers: {selected_message_passing_layers}")
print(f"- selected edge_keep_quantile: {selected_edge_keep_quantile:.2f}")
print(f"- selected refined_directed_edges: {selected_refined_directed_edges}")
print(f"- best_epoch: {NB14_SELECTED_RESULT['best_epoch']}")
print(f"- best_val_loss: {NB14_SELECTED_RESULT['best_val_loss']:.6f}")
print(f"- test_loss: {nb14_selected_test_metrics['loss']:.6f}")
print(f"- test_MAE: {nb14_selected_test_metrics['MAE']:.6f}")
print(f"- test_RMSE: {nb14_selected_test_metrics['RMSE']:.6f}")
print(f"- test_R2: {nb14_selected_test_metrics['R2']:.6f}")
print(f"- test_n_observed_points: {nb14_selected_test_metrics['n_observed_points']}")
print(f"- test evaluated experiment ids: {NB14_TEST_EVALUATED_EXPERIMENT_IDS}")

print("\nNB14 final test summary:")
nb14_test_summary_df

NB14 test-only final reporting:
- selected experiment_id: NB14_E01
- selected message_passing_layers: 2
- selected edge_keep_quantile: 0.50
- selected refined_directed_edges: 534
- best_epoch: 5
- best_val_loss: 0.060545
- test_loss: 0.095326
- test_MAE: 0.218257
- test_RMSE: 0.308930
- test_R2: -0.034714
- test_n_observed_points: 1086336
- test evaluated experiment ids: ['NB14_E01']

NB14 final test summary:


,experiment_id,topology_variant,edge_policy,edge_keep_quantile,message_passing_layers,best_epoch,best_val_loss,best_val_MAE,test_loss,test_MAE,test_RMSE,test_R2,test_n_observed_points,runtime_input_feature_dim,refined_directed_edges,reference_nb13_experiment_id,is_nb13_reference_quantile
0,NB14_E01,local_pruned_graph,keep_shortest_edges_only,0.5,2,5,0.060545,0.179216,0.095326,0.218257,0.30893,-0.034714,1086336,41,534,NB13_E06,True


In [17]:
# NB14 | Strict comparison against NB12 reference and NB13 best run

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Επιστρέφει copy με normalized lowercase column names για ασφαλέστερο matching.
    """
    out = df.copy()
    out.columns = [str(col).strip().lower() for col in out.columns]
    return out


def resolve_required_metric_column(df: pd.DataFrame, candidate_names: list[str], df_name: str) -> str:
    """
    Βρίσκει metric column με fail-fast λογική.
    """
    normalized_cols = {str(col).strip().lower(): col for col in df.columns}

    for candidate in candidate_names:
        candidate_norm = candidate.strip().lower()
        if candidate_norm in normalized_cols:
            return normalized_cols[candidate_norm]

    raise KeyError(
        f"Δεν βρέθηκε metric column στο '{df_name}'. "
        f"Candidates tried: {candidate_names}. "
        f"Available columns: {list(df.columns)}"
    )


def resolve_single_row_metric(
    df: pd.DataFrame,
    metric_candidates: list[str],
    df_name: str,
) -> float:
    """
    Παίρνει metric από single-row dataframe.
    """
    if len(df) != 1:
        raise ValueError(
            f"Το '{df_name}' αναμενόταν να έχει ακριβώς 1 row, αλλά έχει {len(df)}."
        )

    metric_col = resolve_required_metric_column(df, metric_candidates, df_name)
    return float(df.iloc[0][metric_col])


# ------------------------------------------------------------------
# NB12 reference metrics
# ------------------------------------------------------------------
nb12_test_mae = resolve_single_row_metric(
    nb12_test_metrics_df,
    ["test_mae", "mae"],
    "nb12_test_metrics_df",
)
nb12_test_rmse = resolve_single_row_metric(
    nb12_test_metrics_df,
    ["test_rmse", "rmse"],
    "nb12_test_metrics_df",
)
nb12_test_r2 = resolve_single_row_metric(
    nb12_test_metrics_df,
    ["test_r2", "r2"],
    "nb12_test_metrics_df",
)

# ------------------------------------------------------------------
# NB13 best reference metrics
# Χρησιμοποιούμε το experiment id που ήδη πέρασε ως strict NB13 anchor στο NB14.
# ------------------------------------------------------------------
nb13_test_metrics_norm_df = normalize_columns(nb13_test_metrics_df)

if "experiment_id" not in nb13_test_metrics_norm_df.columns:
    raise KeyError(
        "Το nb13_test_metrics_df δεν περιέχει 'experiment_id', άρα δεν μπορεί να γίνει strict NB13 anchor lookup."
    )

nb13_reference_experiment_id = str(NB14_FINAL_TEST_RESULT["reference_nb13_experiment_id"])

nb13_reference_row_df = nb13_test_metrics_norm_df.loc[
    nb13_test_metrics_norm_df["experiment_id"].astype(str) == nb13_reference_experiment_id
].copy()

if len(nb13_reference_row_df) != 1:
    raise ValueError(
        f"Δεν βρέθηκε ακριβώς μία NB13 test row για experiment_id={nb13_reference_experiment_id}. "
        f"Rows found: {len(nb13_reference_row_df)}"
    )

nb13_test_mae = float(
    nb13_reference_row_df.iloc[0][
        resolve_required_metric_column(
            nb13_reference_row_df,
            ["test_mae", "mae"],
            "nb13_reference_row_df",
        )
    ]
)
nb13_test_rmse = float(
    nb13_reference_row_df.iloc[0][
        resolve_required_metric_column(
            nb13_reference_row_df,
            ["test_rmse", "rmse"],
            "nb13_reference_row_df",
        )
    ]
)
nb13_test_r2 = float(
    nb13_reference_row_df.iloc[0][
        resolve_required_metric_column(
            nb13_reference_row_df,
            ["test_r2", "r2"],
            "nb13_reference_row_df",
        )
    ]
)

# ------------------------------------------------------------------
# NB14 selected final test metrics
# ------------------------------------------------------------------
nb14_test_mae = float(NB14_FINAL_TEST_RESULT["test_MAE"])
nb14_test_rmse = float(NB14_FINAL_TEST_RESULT["test_RMSE"])
nb14_test_r2 = float(NB14_FINAL_TEST_RESULT["test_R2"])

# ------------------------------------------------------------------
# Compact comparison table
# ------------------------------------------------------------------
nb14_reference_comparison_df = pd.DataFrame(
    [
        {
            "run_label": "NB12_reference",
            "experiment_id": "NB12_reference",
            "test_MAE": nb12_test_mae,
            "test_RMSE": nb12_test_rmse,
            "test_R2": nb12_test_r2,
        },
        {
            "run_label": "NB13_best_reference",
            "experiment_id": nb13_reference_experiment_id,
            "test_MAE": nb13_test_mae,
            "test_RMSE": nb13_test_rmse,
            "test_R2": nb13_test_r2,
        },
        {
            "run_label": "NB14_selected",
            "experiment_id": str(NB14_FINAL_TEST_RESULT["experiment_id"]),
            "test_MAE": nb14_test_mae,
            "test_RMSE": nb14_test_rmse,
            "test_R2": nb14_test_r2,
        },
    ]
)

nb14_reference_comparison_df["delta_test_MAE_vs_NB12"] = (
    nb14_reference_comparison_df["test_MAE"] - nb12_test_mae
)
nb14_reference_comparison_df["delta_test_MAE_vs_NB13_best"] = (
    nb14_reference_comparison_df["test_MAE"] - nb13_test_mae
)

nb14_reference_comparison_df = nb14_reference_comparison_df.sort_values(
    by=["test_MAE", "run_label"],
    ascending=[True, True],
    kind="mergesort",
).reset_index(drop=True)

# ------------------------------------------------------------------
# Compact textual summary
# ------------------------------------------------------------------
nb14_delta_vs_nb12 = nb14_test_mae - nb12_test_mae
nb14_delta_vs_nb13 = nb14_test_mae - nb13_test_mae

print("Strict NB12 / NB13 / NB14 comparison:")
print(f"- NB12 reference test_MAE: {nb12_test_mae:.6f}")
print(f"- NB13 best reference ({nb13_reference_experiment_id}) test_MAE: {nb13_test_mae:.6f}")
print(f"- NB14 selected ({NB14_FINAL_TEST_RESULT['experiment_id']}) test_MAE: {nb14_test_mae:.6f}")
print(f"- NB14 delta vs NB12 on test_MAE: {nb14_delta_vs_nb12:+.6f}")
print(f"- NB14 delta vs NB13 best on test_MAE: {nb14_delta_vs_nb13:+.6f}")

print("\nNB14 reference comparison table:")
nb14_reference_comparison_df

Strict NB12 / NB13 / NB14 comparison:
- NB12 reference test_MAE: 0.217742
- NB13 best reference (NB13_E06) test_MAE: 0.218184
- NB14 selected (NB14_E01) test_MAE: 0.218257
- NB14 delta vs NB12 on test_MAE: +0.000515
- NB14 delta vs NB13 best on test_MAE: +0.000074

NB14 reference comparison table:


,run_label,experiment_id,test_MAE,test_RMSE,test_R2,delta_test_MAE_vs_NB12,delta_test_MAE_vs_NB13_best
0,NB12_reference,NB12_reference,0.217742,0.309262,-0.036943,0.000000,-0.000441
1,NB13_best_reference,NB13_E06,0.218184,0.308976,-0.035027,0.000441,0.000000
2,NB14_selected,NB14_E01,0.218257,0.308930,-0.034714,0.000515,0.000074


In [18]:
# NB14 | Final sanity test
# Ο στόχος εδώ είναι να επιβεβαιωθεί ότι το notebook παρέμεινε benchmark-safe.

def sanity_assert(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)


print("Running NB14 final sanity test...")

# ------------------------------------------------------------------
# 1) Registry / results integrity
# ------------------------------------------------------------------
sanity_assert(len(NB14_EXPERIMENT_REGISTRY) >= 3, "Το NB14_EXPERIMENT_REGISTRY είναι υπερβολικά μικρό.")
sanity_assert(
    set(NB14_EXPERIMENT_REGISTRY["experiment_id"].astype(str)) == set(NB14_EXPERIMENT_RESULTS.keys()),
    "Ασυμφωνία ανάμεσα στο experiment registry και στα stored validation results."
)

# ------------------------------------------------------------------
# 2) Validation-only selection integrity
# ------------------------------------------------------------------
sanity_assert(len(nb14_validation_summary_df) == len(NB14_EXPERIMENT_REGISTRY), "Ασυμφωνία πλήθους rows στο validation summary.")
sanity_assert(
    "is_selected_for_final_test" in nb14_validation_summary_df.columns,
    "Λείπει η στήλη is_selected_for_final_test από το validation summary."
)
sanity_assert(
    int(nb14_validation_summary_df["is_selected_for_final_test"].sum()) == 1,
    "Πρέπει να υπάρχει ακριβώς ένα selected experiment για final test."
)

selected_validation_row = nb14_validation_summary_df.loc[
    nb14_validation_summary_df["is_selected_for_final_test"]
].iloc[0]

sanity_assert(
    str(selected_validation_row["experiment_id"]) == str(NB14_SELECTED_EXPERIMENT_ID),
    "Το selected row του validation summary δεν συμφωνεί με το NB14_SELECTED_EXPERIMENT_ID."
)
sanity_assert(
    str(nb14_validation_summary_df.iloc[0]["experiment_id"]) == str(NB14_SELECTED_EXPERIMENT_ID),
    "Το selected experiment δεν είναι το top-ranked row του validation summary."
)

# Το validation summary δεν πρέπει να περιέχει final test metrics.
for forbidden_col in ["test_loss", "test_MAE", "test_RMSE", "test_R2"]:
    sanity_assert(
        forbidden_col not in nb14_validation_summary_df.columns,
        f"Το validation summary δεν πρέπει να περιέχει test metric column: {forbidden_col}"
    )

# ------------------------------------------------------------------
# 3) Test-only final reporting integrity
# ------------------------------------------------------------------
sanity_assert(
    NB14_TEST_EVALUATION_COMPLETED is True,
    "Το NB14_TEST_EVALUATION_COMPLETED πρέπει να είναι True."
)
sanity_assert(
    isinstance(NB14_TEST_EVALUATED_EXPERIMENT_IDS, list),
    "Το NB14_TEST_EVALUATED_EXPERIMENT_IDS πρέπει να είναι list."
)
sanity_assert(
    len(NB14_TEST_EVALUATED_EXPERIMENT_IDS) == 1,
    "Πρέπει να έχει αξιολογηθεί στο test ακριβώς ένα experiment."
)
sanity_assert(
    str(NB14_TEST_EVALUATED_EXPERIMENT_IDS[0]) == str(NB14_SELECTED_EXPERIMENT_ID),
    "Το μοναδικό test-evaluated experiment πρέπει να είναι το selected experiment."
)

sanity_assert(len(nb14_test_summary_df) == 1, "Το nb14_test_summary_df πρέπει να έχει ακριβώς 1 row.")
sanity_assert(
    str(nb14_test_summary_df.iloc[0]["experiment_id"]) == str(NB14_SELECTED_EXPERIMENT_ID),
    "Το nb14_test_summary_df δεν αντιστοιχεί στο selected experiment."
)
sanity_assert(
    str(NB14_FINAL_TEST_RESULT["experiment_id"]) == str(NB14_SELECTED_EXPERIMENT_ID),
    "Το NB14_FINAL_TEST_RESULT δεν αντιστοιχεί στο selected experiment."
)

# ------------------------------------------------------------------
# 4) Metric consistency checks
# ------------------------------------------------------------------
sanity_assert(
    np.isfinite(float(NB14_FINAL_TEST_RESULT["test_loss"])),
    "Το final test_loss δεν είναι finite."
)
sanity_assert(
    np.isfinite(float(NB14_FINAL_TEST_RESULT["test_MAE"])),
    "Το final test_MAE δεν είναι finite."
)
sanity_assert(
    np.isfinite(float(NB14_FINAL_TEST_RESULT["test_RMSE"])),
    "Το final test_RMSE δεν είναι finite."
)
sanity_assert(
    np.isfinite(float(NB14_FINAL_TEST_RESULT["test_R2"])),
    "Το final test_R2 δεν είναι finite."
)
sanity_assert(
    int(NB14_FINAL_TEST_RESULT["test_n_observed_points"]) > 0,
    "Το final test_n_observed_points πρέπει να είναι θετικό."
)

sanity_assert(
    np.isclose(
        float(NB14_FINAL_TEST_RESULT["best_val_loss"]),
        float(NB14_SELECTED_RESULT["best_val_loss"]),
        atol=1e-12,
    ),
    "Ασυμφωνία best_val_loss ανάμεσα στο final test result και στο selected validation result."
)

# ------------------------------------------------------------------
# 5) Runtime contract consistency
# ------------------------------------------------------------------
sanity_assert(
    NB14_RUNTIME_STABILIZATION_APPLIED is True,
    "Το runtime stabilization flag πρέπει να είναι True."
)
sanity_assert(
    nb14_validation_summary_df["runtime_input_feature_dim"].nunique() == 1,
    "Το validation summary έχει μη μοναδικό runtime_input_feature_dim."
)
sanity_assert(
    int(nb14_validation_summary_df["runtime_input_feature_dim"].iloc[0]) == int(NB14_RUNTIME_INPUT_FEATURE_DIM),
    "Ασυμφωνία runtime_input_feature_dim ανάμεσα στο summary και στο runtime contract."
)
sanity_assert(
    int(NB14_FINAL_TEST_RESULT["runtime_input_feature_dim"]) == int(NB14_RUNTIME_INPUT_FEATURE_DIM),
    "Ασυμφωνία runtime_input_feature_dim στο final test result."
)

# ------------------------------------------------------------------
# 6) Strict reference-comparison integrity
# ------------------------------------------------------------------
sanity_assert(len(nb14_reference_comparison_df) == 3, "Το reference comparison table πρέπει να έχει ακριβώς 3 rows.")
sanity_assert(
    set(nb14_reference_comparison_df["run_label"]) == {"NB12_reference", "NB13_best_reference", "NB14_selected"},
    "Το reference comparison table δεν περιέχει ακριβώς τα επιτρεπτά run labels."
)

nb14_selected_comparison_row = nb14_reference_comparison_df.loc[
    nb14_reference_comparison_df["run_label"] == "NB14_selected"
].iloc[0]

sanity_assert(
    str(nb14_selected_comparison_row["experiment_id"]) == str(NB14_SELECTED_EXPERIMENT_ID),
    "Η NB14_selected row του comparison table δεν δείχνει στο selected experiment."
)
sanity_assert(
    np.isclose(float(nb14_selected_comparison_row["test_MAE"]), float(NB14_FINAL_TEST_RESULT["test_MAE"]), atol=1e-12),
    "Ασυμφωνία test_MAE ανάμεσα στο comparison table και στο final test result."
)
sanity_assert(
    np.isclose(float(nb14_selected_comparison_row["test_RMSE"]), float(NB14_FINAL_TEST_RESULT["test_RMSE"]), atol=1e-12),
    "Ασυμφωνία test_RMSE ανάμεσα στο comparison table και στο final test result."
)
sanity_assert(
    np.isclose(float(nb14_selected_comparison_row["test_R2"]), float(NB14_FINAL_TEST_RESULT["test_R2"]), atol=1e-12),
    "Ασυμφωνία test_R2 ανάμεσα στο comparison table και στο final test result."
)

print("NB14 FINAL SANITY CHECK PASSED")
print(f"- selected experiment_id: {NB14_SELECTED_EXPERIMENT_ID}")
print(f"- test evaluated experiment ids: {NB14_TEST_EVALUATED_EXPERIMENT_IDS}")
print(f"- runtime_input_feature_dim: {NB14_RUNTIME_INPUT_FEATURE_DIM}")
print(f"- final test_MAE: {float(NB14_FINAL_TEST_RESULT['test_MAE']):.6f}")

Running NB14 final sanity test...
NB14 FINAL SANITY CHECK PASSED
- selected experiment_id: NB14_E01
- test evaluated experiment ids: ['NB14_E01']
- runtime_input_feature_dim: 41
- final test_MAE: 0.218257


## Τελικό συμπέρασμα του `NB14`

Το `NB14` ολοκλήρωσε ένα **controlled graph refinement follow-up** πάνω στη `local_pruned_graph` οικογένεια, χωρίς να αλλάξει το upstream packaging contract, τη model family ή το benchmark selection / reporting boundary.

### Τι ελέγχθηκε εδώ

Το notebook εξέτασε αν ένα μικρό refinement γύρω από το best `NB13` local-pruned reference παραμένει εμπειρικά συνεπές όταν μεταβάλλεται ελεγχόμενα το `edge_keep_quantile`, κρατώντας σταθερά:

- το packaged graph runtime contract
- τη general GCN family
- το validation-only model selection
- το test-only final reporting

### Κύριο empirical εύρημα

Στο παρόν controlled refinement space, το validation-selected `NB14` run ήταν το:

- `NB14_E01`
- με `edge_keep_quantile = 0.50`
- δηλαδή ουσιαστικά το ίδιο quantile anchor με το referenced best `NB13` local-pruned run

Στο final test reporting, το selected `NB14` run δεν παρείχε βελτίωση έναντι των strict comparison anchors:

- παρέμεινε ελαφρώς χειρότερο από το `NB12` reference στο `test_MAE`
- παρέμεινε επίσης οριακά χειρότερο από το `NB13` best reference run

### Ερμηνεία

Άρα, στο scope του `NB14`, **δεν προκύπτει ένδειξη ότι το συγκεκριμένο pruning-strength refinement βελτιώνει το benchmark outcome**.

Το αποτέλεσμα αυτό πρέπει να διαβαστεί προσεκτικά:

- δεν απορρίπτει γενικά τα graph approaches
- δεν αποδεικνύει superiority της `local_pruned_graph` family
- δεν δικαιολογεί άνοιγμα νέου broad graph benchmark μόνο από αυτό το notebook

Αντίθετα, δείχνει ότι:

- η καλύτερη λύση μέσα στο συγκεκριμένο refinement neighborhood παραμένει κοντά στο ήδη γνωστό `NB13` anchor
- οι observed διαφοροποιήσεις είναι μικρές
- και το controlled refinement, σε αυτή τη μορφή, δεν άλλαξε ουσιαστικά το τελικό benchmark ranking

### Σημαντική τεχνική σημείωση

Κατά το runtime materialization εντοπίστηκε ότι το actual packaged payload λειτουργεί με:

- `runtime_input_feature_dim = 41`

ενώ ένα compact `NB12` reference scalar ανέφερε `42`.

Στο `NB14`, το actual packaged payload αντιμετωπίστηκε ως **runtime authority**, και εφαρμόστηκε numerical stabilization μόνο στο already-masked-out unobserved space. Η παρέμβαση αυτή δεν άλλαξε το benchmark contract και επιβεβαιώθηκε από το final sanity test.

### Τι δεν ισχυρίζεται το `NB14`

Το notebook δεν ισχυρίζεται ότι:

- τα graph models είναι συνολικά καλύτερα από το broader forecasting backbone
- το pruning refinement λύνει το graph performance gap
- το `NB14` αποτελεί νέο architecture benchmark
- το παρόν αποτέλεσμα επεκτείνεται σε sequence models, PHM pipelines ή digital twin application logic

### Τελική αποτίμηση

Το `NB14` είναι επιτυχές ως **μεθοδολογικά καθαρό, benchmark-safe και non-overclaiming refinement notebook**.

Η βασική συμβολή του δεν είναι ένα νέο best result, αλλά ότι:

- έλεγξε αυστηρά ένα μόνο refinement question
- επιβεβαίωσε την περιορισμένη εμπειρική επίδραση του pruning-strength neighborhood
- και παρήγαγε thesis-ready evidence ότι το συγκεκριμένο controlled refinement δεν επαρκεί, από μόνο του, για να μεταβάλει ουσιαστικά το benchmark outcome.